# 02. Exploratory Data Analysis (EDA)
การสำรวจและวิเคราะห์สหสัมพันธ์ (Correlation) ระหว่างปัจจัยมหภาคและดัชนีตลาดโลก 17 ตัว กับหุ้นกลุ่มนำเข้าและส่งออก

# EDA

## อัตราแลกเปลี่ยนเงินบาท

### นำเข้า

### โหลดข้อมูลที่คลีนแล้วจาก data/02_intermediate/ และเตรียมฟังก์ชันระบบ

In [ ]:
# ==============================================================================
# โหลดข้อมูลคลีนจาก data/02_intermediate/ และเตรียมฟังก์ชันสำหรับการวิเคราะห์ EDA
# ทำให้ Notebook นี้สามารถรันเดี่ยวๆ ได้อย่างสมบูรณ์ (Modular Execution)
# ==============================================================================
import os
import glob
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns
from IPython.display import display

# ตั้งค่าฟอนต์ภาษาไทยสำหรับการแสดงผลบนกราฟ
def get_thai_font():
    font_candidates = [
        './Sarabun-Regular.ttf',
        '../Sarabun-Regular.ttf',
        '/System/Library/Fonts/Supplemental/Thonburi.ttc',
        '/System/Library/Fonts/Thonburi.ttc',
        '/Library/Fonts/Arial Unicode.ttf',
        '/usr/share/fonts/truetype/sarabun/Sarabun-Regular.ttf'
    ]
    for p in font_candidates:
        if Path(p).exists():
            return fm.FontProperties(fname=p)
    return fm.FontProperties()

thai_font = get_thai_font()
plt.rcParams['font.family'] = thai_font.get_name() if thai_font.get_name() else 'sans-serif'
plt.rcParams['axes.unicode_minus'] = False
sns.set_theme(style="whitegrid")

# โหลดข้อมูลคลีนจาก data/02_intermediate/
intermediate_candidates = [
    Path('../data/02_intermediate'),
    Path('data/02_intermediate'),
    Path('./data/02_intermediate')
]
inter_dir = None
for c in intermediate_candidates:
    if c.exists():
        inter_dir = c.resolve()
        break

if inter_dir:
    for f in sorted(inter_dir.glob('df_*.csv')):
        globals()[f.stem] = pd.read_csv(f)
    print(f"✓ โหลดชุดข้อมูลคลีนจาก {inter_dir} สำเร็จแล้ว (รวม 37 ชุดข้อมูล: 20 หุ้น + 17 ปัจจัยมหภาค)")
else:
    print("⚠ ไม่พบโฟลเดอร์ data/02_intermediate กรุณารัน 01_data_cleaning.ipynb ก่อน")

# ฟังก์ชันทำความสะอาดและแปลงผลตอบแทน (% Change) เป็น float numeric
def clean_pct_series(df):
    df_clean = df.copy()
    df_clean['Date'] = pd.to_datetime(df_clean['Date'])
    df_clean = df_clean.sort_values('Date')
    if 'Pct_Change' in df_clean.columns:
        if df_clean['Pct_Change'].dtype == 'object':
            df_clean['Pct_Change_Numeric'] = df_clean['Pct_Change'].str.rstrip('%').astype('float') / 100.0
        else:
            df_clean['Pct_Change_Numeric'] = pd.to_numeric(df_clean['Pct_Change'], errors='coerce')
    else:
        df_clean['Pct_Change_Numeric'] = df_clean['Close'].pct_change()
    return df_clean

# ฟังก์ชันวิเคราะห์ Correlation และสร้าง Scatter Matrix พร้อม Regression Line สำเร็จรูป
def analyze_correlation_generic(df_macro, df_assets, date_col='Date', macro_col='Pct_Change',
                               asset_col='Pct_Change', group_col='Stock', macro_name='Macro',
                               asset_type='Asset', color='#2ca02c', figsize=(16, 24)):
    thai_font = get_thai_font()
    macro_renamed = f"{macro_name}_{macro_col}"
    df_m = df_macro[[date_col, macro_col]].dropna().rename(columns={macro_col: macro_renamed}).copy()
    df_m[date_col] = pd.to_datetime(df_m[date_col])

    unique_groups = df_assets[group_col].unique()
    num_groups = len(unique_groups)
    num_cols = 2
    num_rows = int(np.ceil(num_groups / num_cols))

    saved_correlations = []
    fig, axes = plt.subplots(num_rows, num_cols, figsize=figsize)
    if num_groups == 1:
        axes = np.array([axes])
    else:
        axes = axes.flatten()

    for idx, group_name in enumerate(unique_groups):
        if idx >= len(axes):
            break
        asset_renamed = f"{group_name}_{asset_col}"
        df_sub = df_assets[df_assets[group_col] == group_name][[date_col, asset_col]].dropna().rename(columns={asset_col: asset_renamed}).copy()
        df_sub[date_col] = pd.to_datetime(df_sub[date_col])

        df_merged = pd.merge(df_m, df_sub, on=date_col, how='inner').sort_values(date_col)
        if df_merged.empty or len(df_merged) < 2:
            continue

        corr = df_merged[macro_renamed].corr(df_merged[asset_renamed])
        saved_correlations.append({
            'Group': group_name,
            'ประเภทสินทรัพย์': asset_type,
            'Macro Factor': macro_name,
            'Correlation': corr
        })

        sns.regplot(
            data=df_merged,
            x=macro_renamed,
            y=asset_renamed,
            ax=axes[idx],
            scatter_kws={'alpha': 0.25, 'color': color, 's': 12},
            line_kws={'color': '#d62728', 'linewidth': 1.5}
        )
        axes[idx].set_title(f"{group_name} (Corr = {corr:.3f})", fontproperties=thai_font, fontsize=12, fontweight='bold')
        axes[idx].set_xlabel(f"{macro_name} ({macro_col})", fontproperties=thai_font, fontsize=11)
        axes[idx].set_ylabel(f"{group_name} ({asset_col})", fontproperties=thai_font, fontsize=11)

    for j in range(idx + 1, len(axes)):
        fig.delaxes(axes[j])

    fig.suptitle(f"Correlation between {macro_name} and {asset_type}", fontproperties=thai_font, fontsize=16, fontweight='bold', y=1.005)
    plt.tight_layout()
    plt.show()

    df_res = pd.DataFrame(saved_correlations).sort_values(by='Correlation', ascending=True).reset_index(drop=True)
    print(f"\n=== [ตารางสรุป] ค่า Correlation ระหว่าง {macro_name} และ {asset_type} (เรียงจากน้อยไปมาก) ===")
    display(df_res)
    return df_res


In [ ]:
import pandas as pd
import numpy as np

# ปรับแต่งข้อมูลเปอร์เซ็นต์ (Pct_Change) ให้เป็น Float
def clean_pct_series(df):
    df_clean = df.copy()
    if 'Pct_Change' in df_clean.columns and df_clean['Pct_Change'].dtype == 'object':
        df_clean['Pct_Change_Numeric'] = df_clean['Pct_Change'].str.rstrip('%').astype('float') / 100.0
    else:
        df_clean['Pct_Change_Numeric'] = pd.to_numeric(df_clean['Pct_Change'], errors='coerce')
    return df_clean

# 1. จัดทำข้อมูล USD_THB ในรูปแบบเปอร์เซ็นต์
df_usd_thb_pct = clean_pct_series(df_USD_THB)
df_macro_input = df_usd_thb_pct[['Date', 'Pct_Change_Numeric']].rename(columns={'Pct_Change_Numeric': 'Pct_Change'})

# 2. รวมข้อมูลหุ้นนำเข้าแต่ละตัวจากรายชื่อ พร้อมระบุชื่อหุ้น (Stock)
import_stock_dfs = {
    'PTTGC': df_PTTGC, 'TOP': df_TOP, 'IRPC': df_IRPC, 'IVL': df_IVL, 'COM7': df_COM7,
    'BGRIM': df_BGRIM, 'GPSC': df_GPSC, 'EGCO': df_EGCO, 'RATCH': df_RATCH, 'BCP': df_BCP
}

merged_import_list = []
for name, df_stock in import_stock_dfs.items():
    if df_stock is not None and not df_stock.empty:
        df_temp = df_stock.copy()
        df_temp['Stock'] = name
        merged_import_list.append(df_temp)

df_import_all_with_stock = pd.concat(merged_import_list, ignore_index=True)

# 3. จัดทำข้อมูลหุ้นนำเข้าในรูปแบบเปอร์เซ็นต์
df_import_all_pct = clean_pct_series(df_import_all_with_stock)
df_assets_input = df_import_all_pct[['Date', 'Pct_Change_Numeric', 'Stock']].rename(columns={'Pct_Change_Numeric': 'Pct_Change'})

# 4. เรียกใช้งานฟังก์ชันสำเร็จรูป analyze_correlation_generic ที่สร้างไว้ทันที
df_summary_result = analyze_correlation_generic(
    df_macro=df_macro_input,
    df_assets=df_assets_input,
    date_col='Date',
    macro_col='Pct_Change',
    asset_col='Pct_Change',
    group_col='Stock',
    macro_name='USD_THB',
    asset_type='หุ้นนำเข้า'
)

### ส่งออก

In [ ]:
import pandas as pd
import numpy as np

# ปรับแต่งข้อมูลเปอร์เซ็นต์ (Pct_Change) ให้เป็น Float
def clean_pct_series(df):
    df_clean = df.copy()
    if 'Pct_Change' in df_clean.columns and df_clean['Pct_Change'].dtype == 'object':
        df_clean['Pct_Change_Numeric'] = df_clean['Pct_Change'].str.rstrip('%').astype('float') / 100.0
    else:
        df_clean['Pct_Change_Numeric'] = pd.to_numeric(df_clean['Pct_Change'], errors='coerce')
    return df_clean

# 1. จัดทำข้อมูล USD_THB ในรูปแบบเปอร์เซ็นต์
df_usd_thb_pct = clean_pct_series(df_USD_THB)
df_macro_input = df_usd_thb_pct[['Date', 'Pct_Change_Numeric']].rename(columns={'Pct_Change_Numeric': 'Pct_Change'})

# 2. รวมข้อมูลหุ้นส่งออกแต่ละตัวจากรายชื่อ พร้อมระบุชื่อหุ้น (Stock)
export_stock_dfs = {
    'DELTA': df_DELTA, 'HANA': df_HANA, 'KCE': df_KCE, 'CPF': df_CPF, 'TU': df_TU,
    'AAI': df_AAI, 'STA': df_STA, 'BDMS': df_BDMS, 'BH': df_BH, 'ITC': df_ITC
}

merged_export_list = []
for name, df_stock in export_stock_dfs.items():
    if df_stock is not None and not df_stock.empty:
        df_temp = df_stock.copy()
        df_temp['Stock'] = name
        merged_export_list.append(df_temp)

df_export_all_with_stock = pd.concat(merged_export_list, ignore_index=True)

# 3. จัดทำข้อมูลหุ้นส่งออกในรูปแบบเปอร์เซ็นต์
df_export_all_pct = clean_pct_series(df_export_all_with_stock)
df_assets_input = df_export_all_pct[['Date', 'Pct_Change_Numeric', 'Stock']].rename(columns={'Pct_Change_Numeric': 'Pct_Change'})

# 4. เรียกใช้งานฟังก์ชันสำเร็จรูป analyze_correlation_generic พร้อมกำหนดสีน้ำเงิน/ฟ้า (#1f77b4)
df_summary_result = analyze_correlation_generic(
    df_macro=df_macro_input,
    df_assets=df_assets_input,
    date_col='Date',
    macro_col='Pct_Change',
    asset_col='Pct_Change',
    group_col='Stock',
    macro_name='USD_THB',
    asset_type='หุ้นส่งออก',
    color='#1f77b4'
)

## พันธบัตร สหรัฐอเมริกา 10 ปี

### นำเข้า

In [ ]:
import pandas as pd
import numpy as np

# ปรับแต่งข้อมูลเปอร์เซ็นต์ (Pct_Change) ให้เป็น Float
def clean_pct_series(df):
    df_clean = df.copy()
    if 'Pct_Change' in df_clean.columns and df_clean['Pct_Change'].dtype == 'object':
        df_clean['Pct_Change_Numeric'] = df_clean['Pct_Change'].str.rstrip('%').astype('float') / 100.0
    else:
        df_clean['Pct_Change_Numeric'] = pd.to_numeric(df_clean['Pct_Change'], errors='coerce')
    return df_clean

# 1. จัดทำข้อมูล US_10Y ในรูปแบบเปอร์เซ็นต์
df_us_10y_pct = clean_pct_series(df_US_10Y)
df_macro_input = df_us_10y_pct[['Date', 'Pct_Change_Numeric']].rename(columns={'Pct_Change_Numeric': 'Pct_Change'})

# 2. รวมข้อมูลหุ้นนำเข้าแต่ละตัวจากรายชื่อ พร้อมระบุชื่อหุ้น (Stock)
import_stock_dfs = {
    'PTTGC': df_PTTGC, 'TOP': df_TOP, 'IRPC': df_IRPC, 'IVL': df_IVL, 'COM7': df_COM7,
    'BGRIM': df_BGRIM, 'GPSC': df_GPSC, 'EGCO': df_EGCO, 'RATCH': df_RATCH, 'BCP': df_BCP
}

merged_import_list = []
for name, df_stock in import_stock_dfs.items():
    if df_stock is not None and not df_stock.empty:
        df_temp = df_stock.copy()
        df_temp['Stock'] = name
        merged_import_list.append(df_temp)

df_import_all_with_stock = pd.concat(merged_import_list, ignore_index=True)

# 3. จัดทำข้อมูลหุ้นนำเข้าในรูปแบบเปอร์เซ็นต์
df_import_all_pct = clean_pct_series(df_import_all_with_stock)
df_assets_input = df_import_all_pct[['Date', 'Pct_Change_Numeric', 'Stock']].rename(columns={'Pct_Change_Numeric': 'Pct_Change'})

# 4. เรียกใช้งานฟังก์ชันสำเร็จรูป analyze_correlation_generic
df_summary_result = analyze_correlation_generic(
    df_macro=df_macro_input,
    df_assets=df_assets_input,
    date_col='Date',
    macro_col='Pct_Change',
    asset_col='Pct_Change',
    group_col='Stock',
    macro_name='พันธบัตรสหรัฐฯ 10 ปี',
    asset_type='หุ้นนำเข้า'
)

### ส่งออก

In [ ]:
import pandas as pd
import numpy as np

# ปรับแต่งข้อมูลเปอร์เซ็นต์ (Pct_Change) ให้เป็น Float
def clean_pct_series(df):
    df_clean = df.copy()
    if 'Pct_Change' in df_clean.columns and df_clean['Pct_Change'].dtype == 'object':
        df_clean['Pct_Change_Numeric'] = df_clean['Pct_Change'].str.rstrip('%').astype('float') / 100.0
    else:
        df_clean['Pct_Change_Numeric'] = pd.to_numeric(df_clean['Pct_Change'], errors='coerce')
    return df_clean

# 1. จัดทำข้อมูล US_10Y ในรูปแบบเปอร์เซ็นต์
df_us_10y_pct = clean_pct_series(df_US_10Y)
df_macro_input = df_us_10y_pct[['Date', 'Pct_Change_Numeric']].rename(columns={'Pct_Change_Numeric': 'Pct_Change'})

# 2. รวมข้อมูลหุ้นส่งออกแต่ละตัวจากรายชื่อ พร้อมระบุชื่อหุ้น (Stock)
export_stock_dfs = {
    'DELTA': df_DELTA, 'HANA': df_HANA, 'KCE': df_KCE, 'CPF': df_CPF, 'TU': df_TU,
    'AAI': df_AAI, 'STA': df_STA, 'BDMS': df_BDMS, 'BH': df_BH, 'ITC': df_ITC
}

merged_export_list = []
for name, df_stock in export_stock_dfs.items():
    if df_stock is not None and not df_stock.empty:
        df_temp = df_stock.copy()
        df_temp['Stock'] = name
        merged_export_list.append(df_temp)

df_export_all_with_stock = pd.concat(merged_export_list, ignore_index=True)

# 3. จัดทำข้อมูลหุ้นส่งออกในรูปแบบเปอร์เซ็นต์
df_export_all_pct = clean_pct_series(df_export_all_with_stock)
df_assets_input = df_export_all_pct[['Date', 'Pct_Change_Numeric', 'Stock']].rename(columns={'Pct_Change_Numeric': 'Pct_Change'})

# 4. เรียกใช้งานฟังก์ชันสำเร็จรูป analyze_correlation_generic พร้อมกำหนดโทนสีฟ้า (#1f77b4)
df_summary_result = analyze_correlation_generic(
    df_macro=df_macro_input,
    df_assets=df_assets_input,
    date_col='Date',
    macro_col='Pct_Change',
    asset_col='Pct_Change',
    group_col='Stock',
    macro_name='พันธบัตรสหรัฐฯ 10 ปี',
    asset_type='หุ้นส่งออก',
    color='#1f77b4'
)

## พันธบัตร สหรัฐอเมริกา 30 ปี

### นำเข้า

In [ ]:
import pandas as pd
import numpy as np

# ปรับแต่งข้อมูลเปอร์เซ็นต์ (Pct_Change) ให้เป็น Float
def clean_pct_series(df):
    df_clean = df.copy()
    if 'Pct_Change' in df_clean.columns and df_clean['Pct_Change'].dtype == 'object':
        df_clean['Pct_Change_Numeric'] = df_clean['Pct_Change'].str.rstrip('%').astype('float') / 100.0
    else:
        df_clean['Pct_Change_Numeric'] = pd.to_numeric(df_clean['Pct_Change'], errors='coerce')
    return df_clean

# 1. จัดทำข้อมูล US_30Y ในรูปแบบเปอร์เซ็นต์
df_us_30y_pct = clean_pct_series(df_US_30Y)
df_macro_input = df_us_30y_pct[['Date', 'Pct_Change_Numeric']].rename(columns={'Pct_Change_Numeric': 'Pct_Change'})

# 2. รวมข้อมูลหุ้นนำเข้าแต่ละตัวจากรายชื่อ พร้อมระบุชื่อหุ้น (Stock)
import_stock_dfs = {
    'PTTGC': df_PTTGC, 'TOP': df_TOP, 'IRPC': df_IRPC, 'IVL': df_IVL, 'COM7': df_COM7,
    'BGRIM': df_BGRIM, 'GPSC': df_GPSC, 'EGCO': df_EGCO, 'RATCH': df_RATCH, 'BCP': df_BCP
}

merged_import_list = []
for name, df_stock in import_stock_dfs.items():
    if df_stock is not None and not df_stock.empty:
        df_temp = df_stock.copy()
        df_temp['Stock'] = name
        merged_import_list.append(df_temp)

df_import_all_with_stock = pd.concat(merged_import_list, ignore_index=True)

# 3. จัดทำข้อมูลหุ้นนำเข้าในรูปแบบเปอร์เซ็นต์
df_import_all_pct = clean_pct_series(df_import_all_with_stock)
df_assets_input = df_import_all_pct[['Date', 'Pct_Change_Numeric', 'Stock']].rename(columns={'Pct_Change_Numeric': 'Pct_Change'})

# 4. เรียกใช้งานฟังก์ชันสำเร็จรูป analyze_correlation_generic
df_summary_result = analyze_correlation_generic(
    df_macro=df_macro_input,
    df_assets=df_assets_input,
    date_col='Date',
    macro_col='Pct_Change',
    asset_col='Pct_Change',
    group_col='Stock',
    macro_name='พันธบัตรสหรัฐฯ 30 ปี',
    asset_type='หุ้นนำเข้า'
)

### ส่งออก

In [ ]:
import pandas as pd
import numpy as np

# ปรับแต่งข้อมูลเปอร์เซ็นต์ (Pct_Change) ให้เป็น Float
def clean_pct_series(df):
    df_clean = df.copy()
    if 'Pct_Change' in df_clean.columns and df_clean['Pct_Change'].dtype == 'object':
        df_clean['Pct_Change_Numeric'] = df_clean['Pct_Change'].str.rstrip('%').astype('float') / 100.0
    else:
        df_clean['Pct_Change_Numeric'] = pd.to_numeric(df_clean['Pct_Change'], errors='coerce')
    return df_clean

# 1. จัดทำข้อมูล US_30Y ในรูปแบบเปอร์เซ็นต์
df_us_30y_pct = clean_pct_series(df_US_30Y)
df_macro_input = df_us_30y_pct[['Date', 'Pct_Change_Numeric']].rename(columns={'Pct_Change_Numeric': 'Pct_Change'})

# 2. รวมข้อมูลหุ้นส่งออกแต่ละตัวจากรายชื่อ พร้อมระบุชื่อหุ้น (Stock)
export_stock_dfs = {
    'DELTA': df_DELTA, 'HANA': df_HANA, 'KCE': df_KCE, 'CPF': df_CPF, 'TU': df_TU,
    'AAI': df_AAI, 'STA': df_STA, 'BDMS': df_BDMS, 'BH': df_BH, 'ITC': df_ITC
}

merged_export_list = []
for name, df_stock in export_stock_dfs.items():
    if df_stock is not None and not df_stock.empty:
        df_temp = df_stock.copy()
        df_temp['Stock'] = name
        merged_export_list.append(df_temp)

df_export_all_with_stock = pd.concat(merged_export_list, ignore_index=True)

# 3. จัดทำข้อมูลหุ้นส่งออกในรูปแบบเปอร์เซ็นต์
df_export_all_pct = clean_pct_series(df_export_all_with_stock)
df_assets_input = df_export_all_pct[['Date', 'Pct_Change_Numeric', 'Stock']].rename(columns={'Pct_Change_Numeric': 'Pct_Change'})

# 4. เรียกใช้งานฟังก์ชันสำเร็จรูป analyze_correlation_generic พร้อมกำหนดโทนสีฟ้า (#1f77b4)
df_summary_result = analyze_correlation_generic(
    df_macro=df_macro_input,
    df_assets=df_assets_input,
    date_col='Date',
    macro_col='Pct_Change',
    asset_col='Pct_Change',
    group_col='Stock',
    macro_name='พันธบัตรสหรัฐฯ 30 ปี',
    asset_type='หุ้นส่งออก',
    color='#1f77b4'
)

## ดอกเบี้ยอ้างอิงประจำวันธนาคารแห่งประเทศไทย

### นำเข้า

In [ ]:
import pandas as pd
import numpy as np

# ปรับแต่งข้อมูลเปอร์เซ็นต์ (Pct_Change) ให้เป็น Float
def clean_pct_series(df):
    df_clean = df.copy()
    if 'Pct_Change' in df_clean.columns and df_clean['Pct_Change'].dtype == 'object':
        df_clean['Pct_Change_Numeric'] = df_clean['Pct_Change'].str.rstrip('%').astype('float') / 100.0
    else:
        df_clean['Pct_Change_Numeric'] = pd.to_numeric(df_clean['Pct_Change'], errors='coerce')
    return df_clean

# 1. จัดทำข้อมูล df_TH_RATE ในรูปแบบเปอร์เซ็นต์
if 'df_TH_RATE' in globals() and df_TH_RATE is not None and not df_TH_RATE.empty:
    df_th_rate_pct = clean_pct_series(df_TH_RATE)
    df_macro_input = df_th_rate_pct[['Date', 'Pct_Change_Numeric']].rename(columns={'Pct_Change_Numeric': 'Pct_Change'})

    # 2. รวมข้อมูลหุ้นนำเข้าแต่ละตัวจากรายชื่อ พร้อมระบุชื่อหุ้น (Stock)
    import_stock_dfs = {
        'PTTGC': df_PTTGC, 'TOP': df_TOP, 'IRPC': df_IRPC, 'IVL': df_IVL, 'COM7': df_COM7,
        'BGRIM': df_BGRIM, 'GPSC': df_GPSC, 'EGCO': df_EGCO, 'RATCH': df_RATCH, 'BCP': df_BCP
    }

    merged_import_list = []
    for name, df_stock in import_stock_dfs.items():
        if df_stock is not None and not df_stock.empty:
            df_temp = df_stock.copy()
            df_temp['Stock'] = name
            merged_import_list.append(df_temp)

    df_import_all_with_stock = pd.concat(merged_import_list, ignore_index=True)

    # 3. จัดทำข้อมูลหุ้นนำเข้าในรูปแบบเปอร์เซ็นต์
    df_import_all_pct = clean_pct_series(df_import_all_with_stock)
    df_assets_input = df_import_all_pct[['Date', 'Pct_Change_Numeric', 'Stock']].rename(columns={'Pct_Change_Numeric': 'Pct_Change'})

    # 4. เรียกใช้งานฟังก์ชันสำเร็จรูป analyze_correlation_generic
    df_summary_result = analyze_correlation_generic(
        df_macro=df_macro_input,
        df_assets=df_assets_input,
        date_col='Date',
        macro_col='Pct_Change',
        asset_col='Pct_Change',
        group_col='Stock',
        macro_name='ดอกเบี้ยอ้างอิง ธปท.',
        asset_type='หุ้นนำเข้า'
    )
else:
    print("ไม่พบข้อมูล df_TH_RATE ในระบบ")

### ส่งออก

In [ ]:
import pandas as pd
import numpy as np

# ปรับแต่งข้อมูลเปอร์เซ็นต์ (Pct_Change) ให้เป็น Float
def clean_pct_series(df):
    df_clean = df.copy()
    if 'Pct_Change' in df_clean.columns and df_clean['Pct_Change'].dtype == 'object':
        df_clean['Pct_Change_Numeric'] = df_clean['Pct_Change'].str.rstrip('%').astype('float') / 100.0
    else:
        df_clean['Pct_Change_Numeric'] = pd.to_numeric(df_clean['Pct_Change'], errors='coerce')
    return df_clean

# 1. จัดทำข้อมูล df_TH_RATE ในรูปแบบเปอร์เซ็นต์
if 'df_TH_RATE' in globals() and df_TH_RATE is not None and not df_TH_RATE.empty:
    df_th_rate_pct = clean_pct_series(df_TH_RATE)
    df_macro_input = df_th_rate_pct[['Date', 'Pct_Change_Numeric']].rename(columns={'Pct_Change_Numeric': 'Pct_Change'})

    # 2. รวมข้อมูลหุ้นส่งออกแต่ละตัวจากรายชื่อ พร้อมระบุชื่อหุ้น (Stock)
    export_stock_dfs = {
        'DELTA': df_DELTA, 'HANA': df_HANA, 'KCE': df_KCE, 'CPF': df_CPF, 'TU': df_TU,
        'AAI': df_AAI, 'STA': df_STA, 'BDMS': df_BDMS, 'BH': df_BH, 'ITC': df_ITC
    }

    merged_export_list = []
    for name, df_stock in export_stock_dfs.items():
        if df_stock is not None and not df_stock.empty:
            df_temp = df_stock.copy()
            df_temp['Stock'] = name
            merged_export_list.append(df_temp)

    df_export_all_with_stock = pd.concat(merged_export_list, ignore_index=True)

    # 3. จัดทำข้อมูลหุ้นส่งออกในรูปแบบเปอร์เซ็นต์
    df_export_all_pct = clean_pct_series(df_export_all_with_stock)
    df_assets_input = df_export_all_pct[['Date', 'Pct_Change_Numeric', 'Stock']].rename(columns={'Pct_Change_Numeric': 'Pct_Change'})

    # 4. เรียกใช้งานฟังก์ชันสำเร็จรูป analyze_correlation_generic พร้อมกำหนดโทนสีฟ้า (#1f77b4)
    df_summary_result = analyze_correlation_generic(
        df_macro=df_macro_input,
        df_assets=df_assets_input,
        date_col='Date',
        macro_col='Pct_Change',
        asset_col='Pct_Change',
        group_col='Stock',
        macro_name='ดอกเบี้ยอ้างอิง ธปท.',
        asset_type='หุ้นส่งออก',
        color='#1f77b4'
    )
else:
    print("ไม่พบข้อมูล df_TH_RATE ในระบบ")

## ราคาน้ำดิบWTI

### นำเข้า

In [ ]:
import pandas as pd
import numpy as np

# ปรับแต่งข้อมูลเปอร์เซ็นต์ (Pct_Change) ให้เป็น Float
def clean_pct_series(df):
    df_clean = df.copy()
    if 'Pct_Change' in df_clean.columns and df_clean['Pct_Change'].dtype == 'object':
        df_clean['Pct_Change_Numeric'] = df_clean['Pct_Change'].str.rstrip('%').astype('float') / 100.0
    else:
        df_clean['Pct_Change_Numeric'] = pd.to_numeric(df_clean['Pct_Change'], errors='coerce')
    return df_clean

# 1. จัดทำข้อมูล df_WTI ในรูปแบบเปอร์เซ็นต์
if 'df_WTI' in globals() and df_WTI is not None and not df_WTI.empty:
    df_wti_pct = clean_pct_series(df_WTI)
    df_macro_input = df_wti_pct[['Date', 'Pct_Change_Numeric']].rename(columns={'Pct_Change_Numeric': 'Pct_Change'})

    # 2. รวมข้อมูลหุ้นนำเข้าแต่ละตัวจากรายชื่อ พร้อมระบุชื่อหุ้น (Stock)
    import_stock_dfs = {
        'PTTGC': df_PTTGC, 'TOP': df_TOP, 'IRPC': df_IRPC, 'IVL': df_IVL, 'COM7': df_COM7,
        'BGRIM': df_BGRIM, 'GPSC': df_GPSC, 'EGCO': df_EGCO, 'RATCH': df_RATCH, 'BCP': df_BCP
    }

    merged_import_list = []
    for name, df_stock in import_stock_dfs.items():
        if df_stock is not None and not df_stock.empty:
            df_temp = df_stock.copy()
            df_temp['Stock'] = name
            merged_import_list.append(df_temp)

    df_import_all_with_stock = pd.concat(merged_import_list, ignore_index=True)

    # 3. จัดทำข้อมูลหุ้นนำเข้าในรูปแบบเปอร์เซ็นต์
    df_import_all_pct = clean_pct_series(df_import_all_with_stock)
    df_assets_input = df_import_all_pct[['Date', 'Pct_Change_Numeric', 'Stock']].rename(columns={'Pct_Change_Numeric': 'Pct_Change'})

    # 4. เรียกใช้งานฟังก์ชันสำเร็จรูป analyze_correlation_generic
    df_summary_result = analyze_correlation_generic(
        df_macro=df_macro_input,
        df_assets=df_assets_input,
        date_col='Date',
        macro_col='Pct_Change',
        asset_col='Pct_Change',
        group_col='Stock',
        macro_name='ราคาน้ำดิบ WTI',
        asset_type='หุ้นนำเข้า'
    )
else:
    print("ไม่พบข้อมูล df_WTI ในระบบ")

### ส่งออก

In [ ]:
import pandas as pd
import numpy as np

# ปรับแต่งข้อมูลเปอร์เซ็นต์ (Pct_Change) ให้เป็น Float
def clean_pct_series(df):
    df_clean = df.copy()
    if 'Pct_Change' in df_clean.columns and df_clean['Pct_Change'].dtype == 'object':
        df_clean['Pct_Change_Numeric'] = df_clean['Pct_Change'].str.rstrip('%').astype('float') / 100.0
    else:
        df_clean['Pct_Change_Numeric'] = pd.to_numeric(df_clean['Pct_Change'], errors='coerce')
    return df_clean

# 1. จัดทำข้อมูล df_WTI ในรูปแบบเปอร์เซ็นต์
if 'df_WTI' in globals() and df_WTI is not None and not df_WTI.empty:
    df_wti_pct = clean_pct_series(df_WTI)
    df_macro_input = df_wti_pct[['Date', 'Pct_Change_Numeric']].rename(columns={'Pct_Change_Numeric': 'Pct_Change'})

    # 2. รวมข้อมูลหุ้นส่งออกแต่ละตัวจากรายชื่อ พร้อมระบุชื่อหุ้น (Stock)
    export_stock_dfs = {
        'DELTA': df_DELTA, 'HANA': df_HANA, 'KCE': df_KCE, 'CPF': df_CPF, 'TU': df_TU,
        'AAI': df_AAI, 'STA': df_STA, 'BDMS': df_BDMS, 'BH': df_BH, 'ITC': df_ITC
    }

    merged_export_list = []
    for name, df_stock in export_stock_dfs.items():
        if df_stock is not None and not df_stock.empty:
            df_temp = df_stock.copy()
            df_temp['Stock'] = name
            merged_export_list.append(df_temp)

    df_export_all_with_stock = pd.concat(merged_export_list, ignore_index=True)

    # 3. จัดทำข้อมูลหุ้นส่งออกในรูปแบบเปอร์เซ็นต์
    df_export_all_pct = clean_pct_series(df_export_all_with_stock)
    df_assets_input = df_export_all_pct[['Date', 'Pct_Change_Numeric', 'Stock']].rename(columns={'Pct_Change_Numeric': 'Pct_Change'})

    # 4. เรียกใช้งานฟังก์ชันสำเร็จรูป analyze_correlation_generic พร้อมกำหนดโทนสีฟ้า (#1f77b4)
    df_summary_result = analyze_correlation_generic(
        df_macro=df_macro_input,
        df_assets=df_assets_input,
        date_col='Date',
        macro_col='Pct_Change',
        asset_col='Pct_Change',
        group_col='Stock',
        macro_name='ราคาน้ำดิบ WTI',
        asset_type='หุ้นส่งออก',
        color='#1f77b4'
    )
else:
    print("ไม่พบข้อมูล df_WTI ในระบบ")

## ราคาน้ำดิบเบรนท์

### นำเข้า

In [ ]:
import pandas as pd
import numpy as np

# ปรับแต่งข้อมูลเปอร์เซ็นต์ (Pct_Change) ให้เป็น Float
def clean_pct_series(df):
    df_clean = df.copy()
    if 'Pct_Change' in df_clean.columns and df_clean['Pct_Change'].dtype == 'object':
        df_clean['Pct_Change_Numeric'] = df_clean['Pct_Change'].str.rstrip('%').astype('float') / 100.0
    else:
        df_clean['Pct_Change_Numeric'] = pd.to_numeric(df_clean['Pct_Change'], errors='coerce')
    return df_clean

# 1. จัดทำข้อมูล df_เบรนท์ ในรูปแบบเปอร์เซ็นต์
if 'df_เบรนท์' in globals() and df_เบรนท์ is not None and not df_เบรนท์.empty:
    df_brent_pct = clean_pct_series(df_เบรนท์)
    df_macro_input = df_brent_pct[['Date', 'Pct_Change_Numeric']].rename(columns={'Pct_Change_Numeric': 'Pct_Change'})

    # 2. รวมข้อมูลหุ้นนำเข้าแต่ละตัวจากรายชื่อ พร้อมระบุชื่อหุ้น (Stock)
    import_stock_dfs = {
        'PTTGC': df_PTTGC, 'TOP': df_TOP, 'IRPC': df_IRPC, 'IVL': df_IVL, 'COM7': df_COM7,
        'BGRIM': df_BGRIM, 'GPSC': df_GPSC, 'EGCO': df_EGCO, 'RATCH': df_RATCH, 'BCP': df_BCP
    }

    merged_import_list = []
    for name, df_stock in import_stock_dfs.items():
        if df_stock is not None and not df_stock.empty:
            df_temp = df_stock.copy()
            df_temp['Stock'] = name
            merged_import_list.append(df_temp)

    df_import_all_with_stock = pd.concat(merged_import_list, ignore_index=True)

    # 3. จัดทำข้อมูลหุ้นนำเข้าในรูปแบบเปอร์เซ็นต์
    df_import_all_pct = clean_pct_series(df_import_all_with_stock)
    df_assets_input = df_import_all_pct[['Date', 'Pct_Change_Numeric', 'Stock']].rename(columns={'Pct_Change_Numeric': 'Pct_Change'})

    # 4. เรียกใช้งานฟังก์ชันสำเร็จรูป analyze_correlation_generic
    df_summary_result = analyze_correlation_generic(
        df_macro=df_macro_input,
        df_assets=df_assets_input,
        date_col='Date',
        macro_col='Pct_Change',
        asset_col='Pct_Change',
        group_col='Stock',
        macro_name='ราคาน้ำดิบเบรนท์',
        asset_type='หุ้นนำเข้า'
    )
else:
    print("ไม่พบข้อมูล df_เบรนท์ ในระบบ")

### ส่งออก

In [ ]:
import pandas as pd
import numpy as np

# ปรับแต่งข้อมูลเปอร์เซ็นต์ (Pct_Change) ให้เป็น Float
def clean_pct_series(df):
    df_clean = df.copy()
    if 'Pct_Change' in df_clean.columns and df_clean['Pct_Change'].dtype == 'object':
        df_clean['Pct_Change_Numeric'] = df_clean['Pct_Change'].str.rstrip('%').astype('float') / 100.0
    else:
        df_clean['Pct_Change_Numeric'] = pd.to_numeric(df_clean['Pct_Change'], errors='coerce')
    return df_clean

# 1. จัดทำข้อมูล df_เบรนท์ ในรูปแบบเปอร์เซ็นต์
if 'df_เบรนท์' in globals() and df_เบรนท์ is not None and not df_เบรนท์.empty:
    df_brent_pct = clean_pct_series(df_เบรนท์)
    df_macro_input = df_brent_pct[['Date', 'Pct_Change_Numeric']].rename(columns={'Pct_Change_Numeric': 'Pct_Change'})

    # 2. รวมข้อมูลหุ้นส่งออกแต่ละตัวจากรายชื่อ พร้อมระบุชื่อหุ้น (Stock)
    export_stock_dfs = {
        'DELTA': df_DELTA, 'HANA': df_HANA, 'KCE': df_KCE, 'CPF': df_CPF, 'TU': df_TU,
        'AAI': df_AAI, 'STA': df_STA, 'BDMS': df_BDMS, 'BH': df_BH, 'ITC': df_ITC
    }

    merged_export_list = []
    for name, df_stock in export_stock_dfs.items():
        if df_stock is not None and not df_stock.empty:
            df_temp = df_stock.copy()
            df_temp['Stock'] = name
            merged_export_list.append(df_temp)

    df_export_all_with_stock = pd.concat(merged_export_list, ignore_index=True)

    # 3. จัดทำข้อมูลหุ้นส่งออกในรูปแบบเปอร์เซ็นต์
    df_export_all_pct = clean_pct_series(df_export_all_with_stock)
    df_assets_input = df_export_all_pct[['Date', 'Pct_Change_Numeric', 'Stock']].rename(columns={'Pct_Change_Numeric': 'Pct_Change'})

    # 4. เรียกใช้งานฟังก์ชันสำเร็จรูป analyze_correlation_generic พร้อมกำหนดโทนสีฟ้า (#1f77b4)
    df_summary_result = analyze_correlation_generic(
        df_macro=df_macro_input,
        df_assets=df_assets_input,
        date_col='Date',
        macro_col='Pct_Change',
        asset_col='Pct_Change',
        group_col='Stock',
        macro_name='ราคาน้ำดิบเบรนท์',
        asset_type='หุ้นส่งออก',
        color='#1f77b4'
    )
else:
    print("ไม่พบข้อมูล df_เบรนท์ ในระบบ")

## ผลตอบแทนพันธบัตรไทย 10 ปี

### นำเข้า

In [ ]:
import pandas as pd
import numpy as np

# ปรับแต่งข้อมูลเปอร์เซ็นต์ (Pct_Change) ให้เป็น Float
def clean_pct_series(df):
    df_clean = df.copy()
    if 'Pct_Change' in df_clean.columns and df_clean['Pct_Change'].dtype == 'object':
        df_clean['Pct_Change_Numeric'] = df_clean['Pct_Change'].str.rstrip('%').astype('float') / 100.0
    else:
        df_clean['Pct_Change_Numeric'] = pd.to_numeric(df_clean['Pct_Change'], errors='coerce')
    return df_clean

# 1. จัดทำข้อมูล df_TH_10Y จากหน่วยความจำโดยตรงในรูปแบบเปอร์เซ็นต์
if 'df_TH_10Y' in globals() and df_TH_10Y is not None and not df_TH_10Y.empty:
    df_th_10y_pct = clean_pct_series(df_TH_10Y)
    df_macro_input = df_th_10y_pct[['Date', 'Pct_Change_Numeric']].rename(columns={'Pct_Change_Numeric': 'Pct_Change'})

    # 2. รวมข้อมูลหุ้นนำเข้าแต่ละตัวจากรายชื่อที่มีอยู่ในระบบ พร้อมระบุชื่อหุ้น (Stock)
    import_stock_dfs = {
        'PTTGC': df_PTTGC, 'TOP': df_TOP, 'IRPC': df_IRPC, 'IVL': df_IVL, 'COM7': df_COM7,
        'BGRIM': df_BGRIM, 'GPSC': df_GPSC, 'EGCO': df_EGCO, 'RATCH': df_RATCH, 'BCP': df_BCP
    }

    merged_import_list = []
    for name, df_stock in import_stock_dfs.items():
        if df_stock is not None and not df_stock.empty:
            df_temp = df_stock.copy()
            df_temp['Stock'] = name
            merged_import_list.append(df_temp)

    df_import_all_with_stock = pd.concat(merged_import_list, ignore_index=True)

    # 3. จัดทำข้อมูลหุ้นนำเข้าในรูปแบบเปอร์เซ็นต์
    df_import_all_pct = clean_pct_series(df_import_all_with_stock)
    df_assets_input = df_import_all_pct[['Date', 'Pct_Change_Numeric', 'Stock']].rename(columns={'Pct_Change_Numeric': 'Pct_Change'})

    # 4. เรียกใช้งานฟังก์ชันสำเร็จรูป analyze_correlation_generic
    print("กำลังวิเคราะห์ค่าความสัมพันธ์และสร้างกราฟสำหรับ ผลตอบแทนพันธบัตรไทย 10 ปี...")
    df_summary_result = analyze_correlation_generic(
        df_macro=df_macro_input,
        df_assets=df_assets_input,
        date_col='Date',
        macro_col='Pct_Change',
        asset_col='Pct_Change',
        group_col='Stock',
        macro_name='ผลตอบแทนพันธบัตรไทย 10 ปี',
        asset_type='หุ้นนำเข้า'
    )
else:
    print("ไม่พบข้อมูล df_TH_10Y ในระบบ")

### ส่งออก

In [ ]:
import pandas as pd
import numpy as np

# ปรับแต่งข้อมูลเปอร์เซ็นต์ (Pct_Change) ให้เป็น Float
def clean_pct_series(df):
    df_clean = df.copy()
    if 'Pct_Change' in df_clean.columns and df_clean['Pct_Change'].dtype == 'object':
        df_clean['Pct_Change_Numeric'] = df_clean['Pct_Change'].str.rstrip('%').astype('float') / 100.0
    else:
        df_clean['Pct_Change_Numeric'] = pd.to_numeric(df_clean['Pct_Change'], errors='coerce')
    return df_clean

# 1. จัดทำข้อมูล df_TH_10Y จากหน่วยความจำโดยตรงในรูปแบบเปอร์เซ็นต์
if 'df_TH_10Y' in globals() and df_TH_10Y is not None and not df_TH_10Y.empty:
    df_th_10y_pct = clean_pct_series(df_TH_10Y)
    df_macro_input = df_th_10y_pct[['Date', 'Pct_Change_Numeric']].rename(columns={'Pct_Change_Numeric': 'Pct_Change'})

    # 2. รวมข้อมูลหุ้นส่งออกแต่ละตัวจากรายชื่อ พร้อมระบุชื่อหุ้น (Stock)
    export_stock_dfs = {
        'DELTA': df_DELTA, 'HANA': df_HANA, 'KCE': df_KCE, 'CPF': df_CPF, 'TU': df_TU,
        'AAI': df_AAI, 'STA': df_STA, 'BDMS': df_BDMS, 'BH': df_BH, 'ITC': df_ITC
    }

    merged_export_list = []
    for name, df_stock in export_stock_dfs.items():
        if df_stock is not None and not df_stock.empty:
            df_temp = df_stock.copy()
            df_temp['Stock'] = name
            merged_export_list.append(df_temp)

    df_export_all_with_stock = pd.concat(merged_export_list, ignore_index=True)

    # 3. จัดทำข้อมูลหุ้นส่งออกในรูปแบบเปอร์เซ็นต์
    df_export_all_pct = clean_pct_series(df_export_all_with_stock)
    df_assets_input = df_export_all_pct[['Date', 'Pct_Change_Numeric', 'Stock']].rename(columns={'Pct_Change_Numeric': 'Pct_Change'})

    # 4. เรียกใช้งานฟังก์ชันสำเร็จรูป analyze_correlation_generic พร้อมกำหนดโทนสีฟ้า (#1f77b4)
    print("กำลังวิเคราะห์ค่าความสัมพันธ์และสร้างกราฟสำหรับ ผลตอบแทนพันธบัตรไทย 10 ปี (กลุ่มส่งออก)...")
    df_summary_result = analyze_correlation_generic(
        df_macro=df_macro_input,
        df_assets=df_assets_input,
        date_col='Date',
        macro_col='Pct_Change',
        asset_col='Pct_Change',
        group_col='Stock',
        macro_name='ผลตอบแทนพันธบัตรไทย 10 ปี',
        asset_type='หุ้นส่งออก',
        color='#1f77b4'
    )
else:
    print("ไม่พบข้อมูล df_TH_10Y ในระบบ")

## ผลตอบแทนพันธบัตรไทย 20 ปี

### นำเข้า

In [ ]:
import pandas as pd
import numpy as np

# ปรับแต่งข้อมูลเปอร์เซ็นต์ (Pct_Change) ให้เป็น Float
def clean_pct_series(df):
    df_clean = df.copy()
    if 'Pct_Change' in df_clean.columns and df_clean['Pct_Change'].dtype == 'object':
        df_clean['Pct_Change_Numeric'] = df_clean['Pct_Change'].str.rstrip('%').astype('float') / 100.0
    else:
        df_clean['Pct_Change_Numeric'] = pd.to_numeric(df_clean['Pct_Change'], errors='coerce')
    return df_clean

# 1. จัดทำข้อมูล df_TH_20Y จากหน่วยความจำโดยตรงในรูปแบบเปอร์เซ็นต์
if 'df_TH_20Y' in globals() and df_TH_20Y is not None and not df_TH_20Y.empty:
    df_th_20y_pct = clean_pct_series(df_TH_20Y)
    df_macro_input = df_th_20y_pct[['Date', 'Pct_Change_Numeric']].rename(columns={'Pct_Change_Numeric': 'Pct_Change'})

    # 2. รวมข้อมูลหุ้นนำเข้าแต่ละตัวจากรายชื่อที่มีอยู่ในระบบ พร้อมระบุชื่อหุ้น (Stock)
    import_stock_dfs = {
        'PTTGC': df_PTTGC, 'TOP': df_TOP, 'IRPC': df_IRPC, 'IVL': df_IVL, 'COM7': df_COM7,
        'BGRIM': df_BGRIM, 'GPSC': df_GPSC, 'EGCO': df_EGCO, 'RATCH': df_RATCH, 'BCP': df_BCP
    }

    merged_import_list = []
    for name, df_stock in import_stock_dfs.items():
        if df_stock is not None and not df_stock.empty:
            df_temp = df_stock.copy()
            df_temp['Stock'] = name
            merged_import_list.append(df_temp)

    df_import_all_with_stock = pd.concat(merged_import_list, ignore_index=True)

    # 3. จัดทำข้อมูลหุ้นนำเข้าในรูปแบบเปอร์เซ็นต์
    df_import_all_pct = clean_pct_series(df_import_all_with_stock)
    df_assets_input = df_import_all_pct[['Date', 'Pct_Change_Numeric', 'Stock']].rename(columns={'Pct_Change_Numeric': 'Pct_Change'})

    # 4. เรียกใช้งานฟังก์ชันสำเร็จรูป analyze_correlation_generic
    print("กำลังวิเคราะห์ค่าความสัมพันธ์และสร้างกราฟสำหรับ ผลตอบแทนพันธบัตรไทย 20 ปี (Pct_Change)...")
    df_summary_result = analyze_correlation_generic(
        df_macro=df_macro_input,
        df_assets=df_assets_input,
        date_col='Date',
        macro_col='Pct_Change',
        asset_col='Pct_Change',
        group_col='Stock',
        macro_name='ผลตอบแทนพันธบัตรไทย 20 ปี',
        asset_type='หุ้นนำเข้า'
    )
else:
    print("ไม่พบข้อมูล df_TH_20Y ในระบบ")

### ส่งออก

In [ ]:
import pandas as pd
import numpy as np

# ปรับแต่งข้อมูลเปอร์เซ็นต์ (Pct_Change) ให้เป็น Float
def clean_pct_series(df):
    df_clean = df.copy()
    if 'Pct_Change' in df_clean.columns and df_clean['Pct_Change'].dtype == 'object':
        df_clean['Pct_Change_Numeric'] = df_clean['Pct_Change'].str.rstrip('%').astype('float') / 100.0
    else:
        df_clean['Pct_Change_Numeric'] = pd.to_numeric(df_clean['Pct_Change'], errors='coerce')
    return df_clean

# 1. จัดทำข้อมูล df_TH_20Y จากหน่วยความจำโดยตรงในรูปแบบเปอร์เซ็นต์
if 'df_TH_20Y' in globals() and df_TH_20Y is not None and not df_TH_20Y.empty:
    df_th_20y_pct = clean_pct_series(df_TH_20Y)
    df_macro_input = df_th_20y_pct[['Date', 'Pct_Change_Numeric']].rename(columns={'Pct_Change_Numeric': 'Pct_Change'})

    # 2. รวมข้อมูลหุ้นส่งออกแต่ละตัวจากรายชื่อ พร้อมระบุชื่อหุ้น (Stock)
    export_stock_dfs = {
        'DELTA': df_DELTA, 'HANA': df_HANA, 'KCE': df_KCE, 'CPF': df_CPF, 'TU': df_TU,
        'AAI': df_AAI, 'STA': df_STA, 'BDMS': df_BDMS, 'BH': df_BH, 'ITC': df_ITC
    }

    merged_export_list = []
    for name, df_stock in export_stock_dfs.items():
        if df_stock is not None and not df_stock.empty:
            df_temp = df_stock.copy()
            df_temp['Stock'] = name
            merged_export_list.append(df_temp)

    df_export_all_with_stock = pd.concat(merged_export_list, ignore_index=True)

    # 3. จัดทำข้อมูลหุ้นส่งออกในรูปแบบเปอร์เซ็นต์
    df_export_all_pct = clean_pct_series(df_export_all_with_stock)
    df_assets_input = df_export_all_pct[['Date', 'Pct_Change_Numeric', 'Stock']].rename(columns={'Pct_Change_Numeric': 'Pct_Change'})

    # 4. เรียกใช้งานฟังก์ชันสำเร็จรูป analyze_correlation_generic พร้อมกำหนดโทนสีฟ้า (#1f77b4)
    print("กำลังวิเคราะห์ค่าความสัมพันธ์และสร้างกราฟสำหรับ ผลตอบแทนพันธบัตรไทย 20 ปี (กลุ่มส่งออก)...")
    df_summary_result = analyze_correlation_generic(
        df_macro=df_macro_input,
        df_assets=df_assets_input,
        date_col='Date',
        macro_col='Pct_Change',
        asset_col='Pct_Change',
        group_col='Stock',
        macro_name='ผลตอบแทนพันธบัตรไทย 20 ปี',
        asset_type='หุ้นส่งออก',
        color='#1f77b4'
    )
else:
    print("ไม่พบข้อมูล df_TH_20Y ในระบบ")

## อัตราแลกเปลี่ยนเงินหยวนจีน (CNY/THB)


### นำเข้า


In [ ]:
import pandas as pd
import numpy as np

# 1. จัดทำข้อมูล CNY_THB ในรูปแบบเปอร์เซ็นต์
df_macro_pct = clean_pct_series(df_CNY_THB)
df_macro_input = df_macro_pct[['Date', 'Pct_Change_Numeric']].rename(columns={'Pct_Change_Numeric': 'Pct_Change'})

# 2. รวมข้อมูลหุ้นนำเข้าแต่ละตัวจากรายชื่อ พร้อมระบุชื่อหุ้น (Stock)
import_stock_dfs = {
    'PTTGC': df_PTTGC, 'TOP': df_TOP, 'IRPC': df_IRPC, 'IVL': df_IVL, 'COM7': df_COM7,
    'BGRIM': df_BGRIM, 'GPSC': df_GPSC, 'EGCO': df_EGCO, 'RATCH': df_RATCH, 'BCP': df_BCP
}

merged_import_list = []
for name, df_stock in import_stock_dfs.items():
    if df_stock is not None and not df_stock.empty:
        df_temp = df_stock.copy()
        df_temp['Stock'] = name
        merged_import_list.append(df_temp)

df_import_all_with_stock = pd.concat(merged_import_list, ignore_index=True)

# 3. จัดทำข้อมูลหุ้นนำเข้าในรูปแบบเปอร์เซ็นต์
df_import_all_pct = clean_pct_series(df_import_all_with_stock)
df_assets_input = df_import_all_pct[['Date', 'Pct_Change_Numeric', 'Stock']].rename(columns={'Pct_Change_Numeric': 'Pct_Change'})

# 4. เรียกใช้งานฟังก์ชันสำเร็จรูป analyze_correlation_generic
df_summary_result = analyze_correlation_generic(
    df_macro=df_macro_input,
    df_assets=df_assets_input,
    date_col='Date',
    macro_col='Pct_Change',
    asset_col='Pct_Change',
    group_col='Stock',
    macro_name='CNY_THB',
    asset_type='หุ้นนำเข้า',
    color='#2ca02c'
)


### ส่งออก


In [ ]:
import pandas as pd
import numpy as np

# 1. จัดทำข้อมูล CNY_THB ในรูปแบบเปอร์เซ็นต์
df_macro_pct = clean_pct_series(df_CNY_THB)
df_macro_input = df_macro_pct[['Date', 'Pct_Change_Numeric']].rename(columns={'Pct_Change_Numeric': 'Pct_Change'})

# 2. รวมข้อมูลหุ้นส่งออกแต่ละตัวจากรายชื่อ พร้อมระบุชื่อหุ้น (Stock)
export_stock_dfs = {
    'DELTA': df_DELTA, 'HANA': df_HANA, 'KCE': df_KCE, 'CPF': df_CPF, 'TU': df_TU,
    'AAI': df_AAI, 'STA': df_STA, 'BDMS': df_BDMS, 'BH': df_BH, 'ITC': df_ITC
}

merged_export_list = []
for name, df_stock in export_stock_dfs.items():
    if df_stock is not None and not df_stock.empty:
        df_temp = df_stock.copy()
        df_temp['Stock'] = name
        merged_export_list.append(df_temp)

df_export_all_with_stock = pd.concat(merged_export_list, ignore_index=True)

# 3. จัดทำข้อมูลหุ้นส่งออกในรูปแบบเปอร์เซ็นต์
df_export_all_pct = clean_pct_series(df_export_all_with_stock)
df_assets_input = df_export_all_pct[['Date', 'Pct_Change_Numeric', 'Stock']].rename(columns={'Pct_Change_Numeric': 'Pct_Change'})

# 4. เรียกใช้งานฟังก์ชันสำเร็จรูป analyze_correlation_generic พร้อมกำหนดสีฟ้า (#1f77b4)
df_summary_result = analyze_correlation_generic(
    df_macro=df_macro_input,
    df_assets=df_assets_input,
    date_col='Date',
    macro_col='Pct_Change',
    asset_col='Pct_Change',
    group_col='Stock',
    macro_name='CNY_THB',
    asset_type='หุ้นส่งออก',
    color='#1f77b4'
)


## อัตราแลกเปลี่ยนเงินยูโร (EUR/THB)


### นำเข้า


In [ ]:
import pandas as pd
import numpy as np

# 1. จัดทำข้อมูล EUR_THB ในรูปแบบเปอร์เซ็นต์
df_macro_pct = clean_pct_series(df_EUR_THB)
df_macro_input = df_macro_pct[['Date', 'Pct_Change_Numeric']].rename(columns={'Pct_Change_Numeric': 'Pct_Change'})

# 2. รวมข้อมูลหุ้นนำเข้าแต่ละตัวจากรายชื่อ พร้อมระบุชื่อหุ้น (Stock)
import_stock_dfs = {
    'PTTGC': df_PTTGC, 'TOP': df_TOP, 'IRPC': df_IRPC, 'IVL': df_IVL, 'COM7': df_COM7,
    'BGRIM': df_BGRIM, 'GPSC': df_GPSC, 'EGCO': df_EGCO, 'RATCH': df_RATCH, 'BCP': df_BCP
}

merged_import_list = []
for name, df_stock in import_stock_dfs.items():
    if df_stock is not None and not df_stock.empty:
        df_temp = df_stock.copy()
        df_temp['Stock'] = name
        merged_import_list.append(df_temp)

df_import_all_with_stock = pd.concat(merged_import_list, ignore_index=True)

# 3. จัดทำข้อมูลหุ้นนำเข้าในรูปแบบเปอร์เซ็นต์
df_import_all_pct = clean_pct_series(df_import_all_with_stock)
df_assets_input = df_import_all_pct[['Date', 'Pct_Change_Numeric', 'Stock']].rename(columns={'Pct_Change_Numeric': 'Pct_Change'})

# 4. เรียกใช้งานฟังก์ชันสำเร็จรูป analyze_correlation_generic
df_summary_result = analyze_correlation_generic(
    df_macro=df_macro_input,
    df_assets=df_assets_input,
    date_col='Date',
    macro_col='Pct_Change',
    asset_col='Pct_Change',
    group_col='Stock',
    macro_name='EUR_THB',
    asset_type='หุ้นนำเข้า',
    color='#2ca02c'
)


### ส่งออก


In [ ]:
import pandas as pd
import numpy as np

# 1. จัดทำข้อมูล EUR_THB ในรูปแบบเปอร์เซ็นต์
df_macro_pct = clean_pct_series(df_EUR_THB)
df_macro_input = df_macro_pct[['Date', 'Pct_Change_Numeric']].rename(columns={'Pct_Change_Numeric': 'Pct_Change'})

# 2. รวมข้อมูลหุ้นส่งออกแต่ละตัวจากรายชื่อ พร้อมระบุชื่อหุ้น (Stock)
export_stock_dfs = {
    'DELTA': df_DELTA, 'HANA': df_HANA, 'KCE': df_KCE, 'CPF': df_CPF, 'TU': df_TU,
    'AAI': df_AAI, 'STA': df_STA, 'BDMS': df_BDMS, 'BH': df_BH, 'ITC': df_ITC
}

merged_export_list = []
for name, df_stock in export_stock_dfs.items():
    if df_stock is not None and not df_stock.empty:
        df_temp = df_stock.copy()
        df_temp['Stock'] = name
        merged_export_list.append(df_temp)

df_export_all_with_stock = pd.concat(merged_export_list, ignore_index=True)

# 3. จัดทำข้อมูลหุ้นส่งออกในรูปแบบเปอร์เซ็นต์
df_export_all_pct = clean_pct_series(df_export_all_with_stock)
df_assets_input = df_export_all_pct[['Date', 'Pct_Change_Numeric', 'Stock']].rename(columns={'Pct_Change_Numeric': 'Pct_Change'})

# 4. เรียกใช้งานฟังก์ชันสำเร็จรูป analyze_correlation_generic พร้อมกำหนดสีฟ้า (#1f77b4)
df_summary_result = analyze_correlation_generic(
    df_macro=df_macro_input,
    df_assets=df_assets_input,
    date_col='Date',
    macro_col='Pct_Change',
    asset_col='Pct_Change',
    group_col='Stock',
    macro_name='EUR_THB',
    asset_type='หุ้นส่งออก',
    color='#1f77b4'
)


## อัตราแลกเปลี่ยนเงินเยนญี่ปุ่น (JPY/THB)


### นำเข้า


In [ ]:
import pandas as pd
import numpy as np

# 1. จัดทำข้อมูล JPY_THB ในรูปแบบเปอร์เซ็นต์
df_macro_pct = clean_pct_series(df_JPY_THB)
df_macro_input = df_macro_pct[['Date', 'Pct_Change_Numeric']].rename(columns={'Pct_Change_Numeric': 'Pct_Change'})

# 2. รวมข้อมูลหุ้นนำเข้าแต่ละตัวจากรายชื่อ พร้อมระบุชื่อหุ้น (Stock)
import_stock_dfs = {
    'PTTGC': df_PTTGC, 'TOP': df_TOP, 'IRPC': df_IRPC, 'IVL': df_IVL, 'COM7': df_COM7,
    'BGRIM': df_BGRIM, 'GPSC': df_GPSC, 'EGCO': df_EGCO, 'RATCH': df_RATCH, 'BCP': df_BCP
}

merged_import_list = []
for name, df_stock in import_stock_dfs.items():
    if df_stock is not None and not df_stock.empty:
        df_temp = df_stock.copy()
        df_temp['Stock'] = name
        merged_import_list.append(df_temp)

df_import_all_with_stock = pd.concat(merged_import_list, ignore_index=True)

# 3. จัดทำข้อมูลหุ้นนำเข้าในรูปแบบเปอร์เซ็นต์
df_import_all_pct = clean_pct_series(df_import_all_with_stock)
df_assets_input = df_import_all_pct[['Date', 'Pct_Change_Numeric', 'Stock']].rename(columns={'Pct_Change_Numeric': 'Pct_Change'})

# 4. เรียกใช้งานฟังก์ชันสำเร็จรูป analyze_correlation_generic
df_summary_result = analyze_correlation_generic(
    df_macro=df_macro_input,
    df_assets=df_assets_input,
    date_col='Date',
    macro_col='Pct_Change',
    asset_col='Pct_Change',
    group_col='Stock',
    macro_name='JPY_THB',
    asset_type='หุ้นนำเข้า',
    color='#2ca02c'
)


### ส่งออก


In [ ]:
import pandas as pd
import numpy as np

# 1. จัดทำข้อมูล JPY_THB ในรูปแบบเปอร์เซ็นต์
df_macro_pct = clean_pct_series(df_JPY_THB)
df_macro_input = df_macro_pct[['Date', 'Pct_Change_Numeric']].rename(columns={'Pct_Change_Numeric': 'Pct_Change'})

# 2. รวมข้อมูลหุ้นส่งออกแต่ละตัวจากรายชื่อ พร้อมระบุชื่อหุ้น (Stock)
export_stock_dfs = {
    'DELTA': df_DELTA, 'HANA': df_HANA, 'KCE': df_KCE, 'CPF': df_CPF, 'TU': df_TU,
    'AAI': df_AAI, 'STA': df_STA, 'BDMS': df_BDMS, 'BH': df_BH, 'ITC': df_ITC
}

merged_export_list = []
for name, df_stock in export_stock_dfs.items():
    if df_stock is not None and not df_stock.empty:
        df_temp = df_stock.copy()
        df_temp['Stock'] = name
        merged_export_list.append(df_temp)

df_export_all_with_stock = pd.concat(merged_export_list, ignore_index=True)

# 3. จัดทำข้อมูลหุ้นส่งออกในรูปแบบเปอร์เซ็นต์
df_export_all_pct = clean_pct_series(df_export_all_with_stock)
df_assets_input = df_export_all_pct[['Date', 'Pct_Change_Numeric', 'Stock']].rename(columns={'Pct_Change_Numeric': 'Pct_Change'})

# 4. เรียกใช้งานฟังก์ชันสำเร็จรูป analyze_correlation_generic พร้อมกำหนดสีฟ้า (#1f77b4)
df_summary_result = analyze_correlation_generic(
    df_macro=df_macro_input,
    df_assets=df_assets_input,
    date_col='Date',
    macro_col='Pct_Change',
    asset_col='Pct_Change',
    group_col='Stock',
    macro_name='JPY_THB',
    asset_type='หุ้นส่งออก',
    color='#1f77b4'
)


## ดัชนีดอลลาร์สหรัฐ (US Dollar Index: DXY)


### นำเข้า


In [ ]:
import pandas as pd
import numpy as np

# 1. จัดทำข้อมูล DXY ในรูปแบบเปอร์เซ็นต์
df_macro_pct = clean_pct_series(df_DXY)
df_macro_input = df_macro_pct[['Date', 'Pct_Change_Numeric']].rename(columns={'Pct_Change_Numeric': 'Pct_Change'})

# 2. รวมข้อมูลหุ้นนำเข้าแต่ละตัวจากรายชื่อ พร้อมระบุชื่อหุ้น (Stock)
import_stock_dfs = {
    'PTTGC': df_PTTGC, 'TOP': df_TOP, 'IRPC': df_IRPC, 'IVL': df_IVL, 'COM7': df_COM7,
    'BGRIM': df_BGRIM, 'GPSC': df_GPSC, 'EGCO': df_EGCO, 'RATCH': df_RATCH, 'BCP': df_BCP
}

merged_import_list = []
for name, df_stock in import_stock_dfs.items():
    if df_stock is not None and not df_stock.empty:
        df_temp = df_stock.copy()
        df_temp['Stock'] = name
        merged_import_list.append(df_temp)

df_import_all_with_stock = pd.concat(merged_import_list, ignore_index=True)

# 3. จัดทำข้อมูลหุ้นนำเข้าในรูปแบบเปอร์เซ็นต์
df_import_all_pct = clean_pct_series(df_import_all_with_stock)
df_assets_input = df_import_all_pct[['Date', 'Pct_Change_Numeric', 'Stock']].rename(columns={'Pct_Change_Numeric': 'Pct_Change'})

# 4. เรียกใช้งานฟังก์ชันสำเร็จรูป analyze_correlation_generic
df_summary_result = analyze_correlation_generic(
    df_macro=df_macro_input,
    df_assets=df_assets_input,
    date_col='Date',
    macro_col='Pct_Change',
    asset_col='Pct_Change',
    group_col='Stock',
    macro_name='DXY',
    asset_type='หุ้นนำเข้า',
    color='#2ca02c'
)


### ส่งออก


In [ ]:
import pandas as pd
import numpy as np

# 1. จัดทำข้อมูล DXY ในรูปแบบเปอร์เซ็นต์
df_macro_pct = clean_pct_series(df_DXY)
df_macro_input = df_macro_pct[['Date', 'Pct_Change_Numeric']].rename(columns={'Pct_Change_Numeric': 'Pct_Change'})

# 2. รวมข้อมูลหุ้นส่งออกแต่ละตัวจากรายชื่อ พร้อมระบุชื่อหุ้น (Stock)
export_stock_dfs = {
    'DELTA': df_DELTA, 'HANA': df_HANA, 'KCE': df_KCE, 'CPF': df_CPF, 'TU': df_TU,
    'AAI': df_AAI, 'STA': df_STA, 'BDMS': df_BDMS, 'BH': df_BH, 'ITC': df_ITC
}

merged_export_list = []
for name, df_stock in export_stock_dfs.items():
    if df_stock is not None and not df_stock.empty:
        df_temp = df_stock.copy()
        df_temp['Stock'] = name
        merged_export_list.append(df_temp)

df_export_all_with_stock = pd.concat(merged_export_list, ignore_index=True)

# 3. จัดทำข้อมูลหุ้นส่งออกในรูปแบบเปอร์เซ็นต์
df_export_all_pct = clean_pct_series(df_export_all_with_stock)
df_assets_input = df_export_all_pct[['Date', 'Pct_Change_Numeric', 'Stock']].rename(columns={'Pct_Change_Numeric': 'Pct_Change'})

# 4. เรียกใช้งานฟังก์ชันสำเร็จรูป analyze_correlation_generic พร้อมกำหนดสีฟ้า (#1f77b4)
df_summary_result = analyze_correlation_generic(
    df_macro=df_macro_input,
    df_assets=df_assets_input,
    date_col='Date',
    macro_col='Pct_Change',
    asset_col='Pct_Change',
    group_col='Stock',
    macro_name='DXY',
    asset_type='หุ้นส่งออก',
    color='#1f77b4'
)


## ดัชนีตลาดหุ้นไทย SET50


### นำเข้า


In [ ]:
import pandas as pd
import numpy as np

# 1. จัดทำข้อมูล SET50 ในรูปแบบเปอร์เซ็นต์
df_macro_pct = clean_pct_series(df_SET50)
df_macro_input = df_macro_pct[['Date', 'Pct_Change_Numeric']].rename(columns={'Pct_Change_Numeric': 'Pct_Change'})

# 2. รวมข้อมูลหุ้นนำเข้าแต่ละตัวจากรายชื่อ พร้อมระบุชื่อหุ้น (Stock)
import_stock_dfs = {
    'PTTGC': df_PTTGC, 'TOP': df_TOP, 'IRPC': df_IRPC, 'IVL': df_IVL, 'COM7': df_COM7,
    'BGRIM': df_BGRIM, 'GPSC': df_GPSC, 'EGCO': df_EGCO, 'RATCH': df_RATCH, 'BCP': df_BCP
}

merged_import_list = []
for name, df_stock in import_stock_dfs.items():
    if df_stock is not None and not df_stock.empty:
        df_temp = df_stock.copy()
        df_temp['Stock'] = name
        merged_import_list.append(df_temp)

df_import_all_with_stock = pd.concat(merged_import_list, ignore_index=True)

# 3. จัดทำข้อมูลหุ้นนำเข้าในรูปแบบเปอร์เซ็นต์
df_import_all_pct = clean_pct_series(df_import_all_with_stock)
df_assets_input = df_import_all_pct[['Date', 'Pct_Change_Numeric', 'Stock']].rename(columns={'Pct_Change_Numeric': 'Pct_Change'})

# 4. เรียกใช้งานฟังก์ชันสำเร็จรูป analyze_correlation_generic
df_summary_result = analyze_correlation_generic(
    df_macro=df_macro_input,
    df_assets=df_assets_input,
    date_col='Date',
    macro_col='Pct_Change',
    asset_col='Pct_Change',
    group_col='Stock',
    macro_name='SET50',
    asset_type='หุ้นนำเข้า',
    color='#2ca02c'
)


### ส่งออก


In [ ]:
import pandas as pd
import numpy as np

# 1. จัดทำข้อมูล SET50 ในรูปแบบเปอร์เซ็นต์
df_macro_pct = clean_pct_series(df_SET50)
df_macro_input = df_macro_pct[['Date', 'Pct_Change_Numeric']].rename(columns={'Pct_Change_Numeric': 'Pct_Change'})

# 2. รวมข้อมูลหุ้นส่งออกแต่ละตัวจากรายชื่อ พร้อมระบุชื่อหุ้น (Stock)
export_stock_dfs = {
    'DELTA': df_DELTA, 'HANA': df_HANA, 'KCE': df_KCE, 'CPF': df_CPF, 'TU': df_TU,
    'AAI': df_AAI, 'STA': df_STA, 'BDMS': df_BDMS, 'BH': df_BH, 'ITC': df_ITC
}

merged_export_list = []
for name, df_stock in export_stock_dfs.items():
    if df_stock is not None and not df_stock.empty:
        df_temp = df_stock.copy()
        df_temp['Stock'] = name
        merged_export_list.append(df_temp)

df_export_all_with_stock = pd.concat(merged_export_list, ignore_index=True)

# 3. จัดทำข้อมูลหุ้นส่งออกในรูปแบบเปอร์เซ็นต์
df_export_all_pct = clean_pct_series(df_export_all_with_stock)
df_assets_input = df_export_all_pct[['Date', 'Pct_Change_Numeric', 'Stock']].rename(columns={'Pct_Change_Numeric': 'Pct_Change'})

# 4. เรียกใช้งานฟังก์ชันสำเร็จรูป analyze_correlation_generic พร้อมกำหนดสีฟ้า (#1f77b4)
df_summary_result = analyze_correlation_generic(
    df_macro=df_macro_input,
    df_assets=df_assets_input,
    date_col='Date',
    macro_col='Pct_Change',
    asset_col='Pct_Change',
    group_col='Stock',
    macro_name='SET50',
    asset_type='หุ้นส่งออก',
    color='#1f77b4'
)


## ดัชนี S&P 500 (ตลาดหุ้นสหรัฐฯ)


### นำเข้า


In [ ]:
import pandas as pd
import numpy as np

# 1. จัดทำข้อมูล SP500 ในรูปแบบเปอร์เซ็นต์
df_macro_pct = clean_pct_series(df_SP500)
df_macro_input = df_macro_pct[['Date', 'Pct_Change_Numeric']].rename(columns={'Pct_Change_Numeric': 'Pct_Change'})

# 2. รวมข้อมูลหุ้นนำเข้าแต่ละตัวจากรายชื่อ พร้อมระบุชื่อหุ้น (Stock)
import_stock_dfs = {
    'PTTGC': df_PTTGC, 'TOP': df_TOP, 'IRPC': df_IRPC, 'IVL': df_IVL, 'COM7': df_COM7,
    'BGRIM': df_BGRIM, 'GPSC': df_GPSC, 'EGCO': df_EGCO, 'RATCH': df_RATCH, 'BCP': df_BCP
}

merged_import_list = []
for name, df_stock in import_stock_dfs.items():
    if df_stock is not None and not df_stock.empty:
        df_temp = df_stock.copy()
        df_temp['Stock'] = name
        merged_import_list.append(df_temp)

df_import_all_with_stock = pd.concat(merged_import_list, ignore_index=True)

# 3. จัดทำข้อมูลหุ้นนำเข้าในรูปแบบเปอร์เซ็นต์
df_import_all_pct = clean_pct_series(df_import_all_with_stock)
df_assets_input = df_import_all_pct[['Date', 'Pct_Change_Numeric', 'Stock']].rename(columns={'Pct_Change_Numeric': 'Pct_Change'})

# 4. เรียกใช้งานฟังก์ชันสำเร็จรูป analyze_correlation_generic
df_summary_result = analyze_correlation_generic(
    df_macro=df_macro_input,
    df_assets=df_assets_input,
    date_col='Date',
    macro_col='Pct_Change',
    asset_col='Pct_Change',
    group_col='Stock',
    macro_name='SP500',
    asset_type='หุ้นนำเข้า',
    color='#2ca02c'
)


### ส่งออก


In [ ]:
import pandas as pd
import numpy as np

# 1. จัดทำข้อมูล SP500 ในรูปแบบเปอร์เซ็นต์
df_macro_pct = clean_pct_series(df_SP500)
df_macro_input = df_macro_pct[['Date', 'Pct_Change_Numeric']].rename(columns={'Pct_Change_Numeric': 'Pct_Change'})

# 2. รวมข้อมูลหุ้นส่งออกแต่ละตัวจากรายชื่อ พร้อมระบุชื่อหุ้น (Stock)
export_stock_dfs = {
    'DELTA': df_DELTA, 'HANA': df_HANA, 'KCE': df_KCE, 'CPF': df_CPF, 'TU': df_TU,
    'AAI': df_AAI, 'STA': df_STA, 'BDMS': df_BDMS, 'BH': df_BH, 'ITC': df_ITC
}

merged_export_list = []
for name, df_stock in export_stock_dfs.items():
    if df_stock is not None and not df_stock.empty:
        df_temp = df_stock.copy()
        df_temp['Stock'] = name
        merged_export_list.append(df_temp)

df_export_all_with_stock = pd.concat(merged_export_list, ignore_index=True)

# 3. จัดทำข้อมูลหุ้นส่งออกในรูปแบบเปอร์เซ็นต์
df_export_all_pct = clean_pct_series(df_export_all_with_stock)
df_assets_input = df_export_all_pct[['Date', 'Pct_Change_Numeric', 'Stock']].rename(columns={'Pct_Change_Numeric': 'Pct_Change'})

# 4. เรียกใช้งานฟังก์ชันสำเร็จรูป analyze_correlation_generic พร้อมกำหนดสีฟ้า (#1f77b4)
df_summary_result = analyze_correlation_generic(
    df_macro=df_macro_input,
    df_assets=df_assets_input,
    date_col='Date',
    macro_col='Pct_Change',
    asset_col='Pct_Change',
    group_col='Stock',
    macro_name='SP500',
    asset_type='หุ้นส่งออก',
    color='#1f77b4'
)


## ดัชนีเซมิคอนดักเตอร์ (Philadelphia Semiconductor Index: SOX)


### นำเข้า


In [ ]:
import pandas as pd
import numpy as np

# 1. จัดทำข้อมูล SOX ในรูปแบบเปอร์เซ็นต์
df_macro_pct = clean_pct_series(df_SOX)
df_macro_input = df_macro_pct[['Date', 'Pct_Change_Numeric']].rename(columns={'Pct_Change_Numeric': 'Pct_Change'})

# 2. รวมข้อมูลหุ้นนำเข้าแต่ละตัวจากรายชื่อ พร้อมระบุชื่อหุ้น (Stock)
import_stock_dfs = {
    'PTTGC': df_PTTGC, 'TOP': df_TOP, 'IRPC': df_IRPC, 'IVL': df_IVL, 'COM7': df_COM7,
    'BGRIM': df_BGRIM, 'GPSC': df_GPSC, 'EGCO': df_EGCO, 'RATCH': df_RATCH, 'BCP': df_BCP
}

merged_import_list = []
for name, df_stock in import_stock_dfs.items():
    if df_stock is not None and not df_stock.empty:
        df_temp = df_stock.copy()
        df_temp['Stock'] = name
        merged_import_list.append(df_temp)

df_import_all_with_stock = pd.concat(merged_import_list, ignore_index=True)

# 3. จัดทำข้อมูลหุ้นนำเข้าในรูปแบบเปอร์เซ็นต์
df_import_all_pct = clean_pct_series(df_import_all_with_stock)
df_assets_input = df_import_all_pct[['Date', 'Pct_Change_Numeric', 'Stock']].rename(columns={'Pct_Change_Numeric': 'Pct_Change'})

# 4. เรียกใช้งานฟังก์ชันสำเร็จรูป analyze_correlation_generic
df_summary_result = analyze_correlation_generic(
    df_macro=df_macro_input,
    df_assets=df_assets_input,
    date_col='Date',
    macro_col='Pct_Change',
    asset_col='Pct_Change',
    group_col='Stock',
    macro_name='SOX',
    asset_type='หุ้นนำเข้า',
    color='#2ca02c'
)


### ส่งออก


In [ ]:
import pandas as pd
import numpy as np

# 1. จัดทำข้อมูล SOX ในรูปแบบเปอร์เซ็นต์
df_macro_pct = clean_pct_series(df_SOX)
df_macro_input = df_macro_pct[['Date', 'Pct_Change_Numeric']].rename(columns={'Pct_Change_Numeric': 'Pct_Change'})

# 2. รวมข้อมูลหุ้นส่งออกแต่ละตัวจากรายชื่อ พร้อมระบุชื่อหุ้น (Stock)
export_stock_dfs = {
    'DELTA': df_DELTA, 'HANA': df_HANA, 'KCE': df_KCE, 'CPF': df_CPF, 'TU': df_TU,
    'AAI': df_AAI, 'STA': df_STA, 'BDMS': df_BDMS, 'BH': df_BH, 'ITC': df_ITC
}

merged_export_list = []
for name, df_stock in export_stock_dfs.items():
    if df_stock is not None and not df_stock.empty:
        df_temp = df_stock.copy()
        df_temp['Stock'] = name
        merged_export_list.append(df_temp)

df_export_all_with_stock = pd.concat(merged_export_list, ignore_index=True)

# 3. จัดทำข้อมูลหุ้นส่งออกในรูปแบบเปอร์เซ็นต์
df_export_all_pct = clean_pct_series(df_export_all_with_stock)
df_assets_input = df_export_all_pct[['Date', 'Pct_Change_Numeric', 'Stock']].rename(columns={'Pct_Change_Numeric': 'Pct_Change'})

# 4. เรียกใช้งานฟังก์ชันสำเร็จรูป analyze_correlation_generic พร้อมกำหนดสีฟ้า (#1f77b4)
df_summary_result = analyze_correlation_generic(
    df_macro=df_macro_input,
    df_assets=df_assets_input,
    date_col='Date',
    macro_col='Pct_Change',
    asset_col='Pct_Change',
    group_col='Stock',
    macro_name='SOX',
    asset_type='หุ้นส่งออก',
    color='#1f77b4'
)


## ดัชนีความผันผวนและความกลัวของตลาดโลก (CBOE Volatility Index: VIX)


### นำเข้า


In [ ]:
import pandas as pd
import numpy as np

# 1. จัดทำข้อมูล VIX ในรูปแบบเปอร์เซ็นต์
df_macro_pct = clean_pct_series(df_VIX)
df_macro_input = df_macro_pct[['Date', 'Pct_Change_Numeric']].rename(columns={'Pct_Change_Numeric': 'Pct_Change'})

# 2. รวมข้อมูลหุ้นนำเข้าแต่ละตัวจากรายชื่อ พร้อมระบุชื่อหุ้น (Stock)
import_stock_dfs = {
    'PTTGC': df_PTTGC, 'TOP': df_TOP, 'IRPC': df_IRPC, 'IVL': df_IVL, 'COM7': df_COM7,
    'BGRIM': df_BGRIM, 'GPSC': df_GPSC, 'EGCO': df_EGCO, 'RATCH': df_RATCH, 'BCP': df_BCP
}

merged_import_list = []
for name, df_stock in import_stock_dfs.items():
    if df_stock is not None and not df_stock.empty:
        df_temp = df_stock.copy()
        df_temp['Stock'] = name
        merged_import_list.append(df_temp)

df_import_all_with_stock = pd.concat(merged_import_list, ignore_index=True)

# 3. จัดทำข้อมูลหุ้นนำเข้าในรูปแบบเปอร์เซ็นต์
df_import_all_pct = clean_pct_series(df_import_all_with_stock)
df_assets_input = df_import_all_pct[['Date', 'Pct_Change_Numeric', 'Stock']].rename(columns={'Pct_Change_Numeric': 'Pct_Change'})

# 4. เรียกใช้งานฟังก์ชันสำเร็จรูป analyze_correlation_generic
df_summary_result = analyze_correlation_generic(
    df_macro=df_macro_input,
    df_assets=df_assets_input,
    date_col='Date',
    macro_col='Pct_Change',
    asset_col='Pct_Change',
    group_col='Stock',
    macro_name='VIX',
    asset_type='หุ้นนำเข้า',
    color='#2ca02c'
)


### ส่งออก


In [ ]:
import pandas as pd
import numpy as np

# 1. จัดทำข้อมูล VIX ในรูปแบบเปอร์เซ็นต์
df_macro_pct = clean_pct_series(df_VIX)
df_macro_input = df_macro_pct[['Date', 'Pct_Change_Numeric']].rename(columns={'Pct_Change_Numeric': 'Pct_Change'})

# 2. รวมข้อมูลหุ้นส่งออกแต่ละตัวจากรายชื่อ พร้อมระบุชื่อหุ้น (Stock)
export_stock_dfs = {
    'DELTA': df_DELTA, 'HANA': df_HANA, 'KCE': df_KCE, 'CPF': df_CPF, 'TU': df_TU,
    'AAI': df_AAI, 'STA': df_STA, 'BDMS': df_BDMS, 'BH': df_BH, 'ITC': df_ITC
}

merged_export_list = []
for name, df_stock in export_stock_dfs.items():
    if df_stock is not None and not df_stock.empty:
        df_temp = df_stock.copy()
        df_temp['Stock'] = name
        merged_export_list.append(df_temp)

df_export_all_with_stock = pd.concat(merged_export_list, ignore_index=True)

# 3. จัดทำข้อมูลหุ้นส่งออกในรูปแบบเปอร์เซ็นต์
df_export_all_pct = clean_pct_series(df_export_all_with_stock)
df_assets_input = df_export_all_pct[['Date', 'Pct_Change_Numeric', 'Stock']].rename(columns={'Pct_Change_Numeric': 'Pct_Change'})

# 4. เรียกใช้งานฟังก์ชันสำเร็จรูป analyze_correlation_generic พร้อมกำหนดสีฟ้า (#1f77b4)
df_summary_result = analyze_correlation_generic(
    df_macro=df_macro_input,
    df_assets=df_assets_input,
    date_col='Date',
    macro_col='Pct_Change',
    asset_col='Pct_Change',
    group_col='Stock',
    macro_name='VIX',
    asset_type='หุ้นส่งออก',
    color='#1f77b4'
)


## ดัชนีค่าระวางเรือเทกอง (Baltic Dry Index: BDI)


### นำเข้า


In [ ]:
import pandas as pd
import numpy as np

# 1. จัดทำข้อมูล BDI ในรูปแบบเปอร์เซ็นต์
df_macro_pct = clean_pct_series(df_BDI)
df_macro_input = df_macro_pct[['Date', 'Pct_Change_Numeric']].rename(columns={'Pct_Change_Numeric': 'Pct_Change'})

# 2. รวมข้อมูลหุ้นนำเข้าแต่ละตัวจากรายชื่อ พร้อมระบุชื่อหุ้น (Stock)
import_stock_dfs = {
    'PTTGC': df_PTTGC, 'TOP': df_TOP, 'IRPC': df_IRPC, 'IVL': df_IVL, 'COM7': df_COM7,
    'BGRIM': df_BGRIM, 'GPSC': df_GPSC, 'EGCO': df_EGCO, 'RATCH': df_RATCH, 'BCP': df_BCP
}

merged_import_list = []
for name, df_stock in import_stock_dfs.items():
    if df_stock is not None and not df_stock.empty:
        df_temp = df_stock.copy()
        df_temp['Stock'] = name
        merged_import_list.append(df_temp)

df_import_all_with_stock = pd.concat(merged_import_list, ignore_index=True)

# 3. จัดทำข้อมูลหุ้นนำเข้าในรูปแบบเปอร์เซ็นต์
df_import_all_pct = clean_pct_series(df_import_all_with_stock)
df_assets_input = df_import_all_pct[['Date', 'Pct_Change_Numeric', 'Stock']].rename(columns={'Pct_Change_Numeric': 'Pct_Change'})

# 4. เรียกใช้งานฟังก์ชันสำเร็จรูป analyze_correlation_generic
df_summary_result = analyze_correlation_generic(
    df_macro=df_macro_input,
    df_assets=df_assets_input,
    date_col='Date',
    macro_col='Pct_Change',
    asset_col='Pct_Change',
    group_col='Stock',
    macro_name='BDI',
    asset_type='หุ้นนำเข้า',
    color='#2ca02c'
)


### ส่งออก


In [ ]:
import pandas as pd
import numpy as np

# 1. จัดทำข้อมูล BDI ในรูปแบบเปอร์เซ็นต์
df_macro_pct = clean_pct_series(df_BDI)
df_macro_input = df_macro_pct[['Date', 'Pct_Change_Numeric']].rename(columns={'Pct_Change_Numeric': 'Pct_Change'})

# 2. รวมข้อมูลหุ้นส่งออกแต่ละตัวจากรายชื่อ พร้อมระบุชื่อหุ้น (Stock)
export_stock_dfs = {
    'DELTA': df_DELTA, 'HANA': df_HANA, 'KCE': df_KCE, 'CPF': df_CPF, 'TU': df_TU,
    'AAI': df_AAI, 'STA': df_STA, 'BDMS': df_BDMS, 'BH': df_BH, 'ITC': df_ITC
}

merged_export_list = []
for name, df_stock in export_stock_dfs.items():
    if df_stock is not None and not df_stock.empty:
        df_temp = df_stock.copy()
        df_temp['Stock'] = name
        merged_export_list.append(df_temp)

df_export_all_with_stock = pd.concat(merged_export_list, ignore_index=True)

# 3. จัดทำข้อมูลหุ้นส่งออกในรูปแบบเปอร์เซ็นต์
df_export_all_pct = clean_pct_series(df_export_all_with_stock)
df_assets_input = df_export_all_pct[['Date', 'Pct_Change_Numeric', 'Stock']].rename(columns={'Pct_Change_Numeric': 'Pct_Change'})

# 4. เรียกใช้งานฟังก์ชันสำเร็จรูป analyze_correlation_generic พร้อมกำหนดสีฟ้า (#1f77b4)
df_summary_result = analyze_correlation_generic(
    df_macro=df_macro_input,
    df_assets=df_assets_input,
    date_col='Date',
    macro_col='Pct_Change',
    asset_col='Pct_Change',
    group_col='Stock',
    macro_name='BDI',
    asset_type='หุ้นส่งออก',
    color='#1f77b4'
)


## Heatmap (สหสัมพันธ์ 17 ปัจจัยมหภาค x หุ้นนำเข้าและส่งออก)


### 1. สหสัมพันธ์ของปัจจัยมหภาค กับ หุ้นกลุ่มนำเข้า (Import Stocks)
การวิเคราะห์ความสัมพันธ์แบบจับคู่ (Pairwise Correlation) ระหว่างผลตอบแทนรายวัน (% Change) ของหุ้นกลุ่มนำเข้า 10 ตัว กับ 17 ปัจจัยมหภาคและดัชนีตลาดโลก

In [ ]:
import os
from pathlib import Path
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from IPython.display import display

# ตั้งค่าฟอนต์ภาษาไทยสำหรับการแสดงผล
def get_thai_font():
    font_candidates = [
        './Sarabun-Regular.ttf',
        '../Sarabun-Regular.ttf',
        '/System/Library/Fonts/Supplemental/Thonburi.ttc',
        '/System/Library/Fonts/Thonburi.ttc',
        '/Library/Fonts/Arial Unicode.ttf',
        '/usr/share/fonts/truetype/sarabun/Sarabun-Regular.ttf'
    ]
    for p in font_candidates:
        if Path(p).exists():
            return fm.FontProperties(fname=p)
    return fm.FontProperties()

thai_font = get_thai_font()

# ฟังก์ชันแปลง % Change ให้เป็นตัวเลขแบบปลอดภัย
def get_clean_pct(df):
    df_c = df.copy()
    df_c['Date'] = pd.to_datetime(df_c['Date'])
    if 'Pct_Change' in df_c.columns:
        if df_c['Pct_Change'].dtype == 'object':
            df_c['Pct_Val'] = df_c['Pct_Change'].str.rstrip('%').astype('float') / 100.0
        else:
            df_c['Pct_Val'] = pd.to_numeric(df_c['Pct_Change'], errors='coerce')
    else:
        df_c = df_c.sort_values('Date')
        df_c['Pct_Val'] = df_c['Close'].pct_change()
    return df_c[['Date', 'Pct_Val']].dropna()

# ฟังก์ชันคำนวณ Pairwise Correlation Matrix สำหรับกลุ่มหุ้นกับปัจจัยมหภาค
def compute_stock_macro_correlation(stocks_dict, macro_dict):
    results = []
    for stock_name, df_s in stocks_dict.items():
        df_s_clean = get_clean_pct(df_s)
        for factor_name, df_f in macro_dict.items():
            df_f_clean = get_clean_pct(df_f)
            merged = pd.merge(df_s_clean, df_f_clean, on='Date', how='inner')
            corr_val = merged['Pct_Val_x'].corr(merged['Pct_Val_y']) if len(merged) >= 5 else np.nan
            results.append({'Stock': stock_name, 'Macro_Factor': factor_name, 'Correlation': corr_val})
    
    df_corr = pd.DataFrame(results).pivot(index='Stock', columns='Macro_Factor', values='Correlation')
    ordered_stocks = [s for s in stocks_dict.keys() if s in df_corr.index]
    ordered_factors = [f for f in macro_dict.keys() if f in df_corr.columns]
    return df_corr.reindex(index=ordered_stocks, columns=ordered_factors)

# รายชื่อกลุ่มหุ้นนำเข้า 10 ตัว
import_stocks_dict = {
    'PTTGC': df_PTTGC, 'TOP': df_TOP, 'IRPC': df_IRPC, 'IVL': df_IVL, 'COM7': df_COM7,
    'BGRIM': df_BGRIM, 'GPSC': df_GPSC, 'EGCO': df_EGCO, 'RATCH': df_RATCH, 'BCP': df_BCP
}

# รายชื่อปัจจัยมหภาคและดัชนีตลาดโลกหลัก 17 ตัว
macro_factors_dict = {
    # อัตราแลกเปลี่ยน
    'USD_THB': df_USD_THB,
    'CNY_THB': df_CNY_THB,
    'EUR_THB': df_EUR_THB,
    'JPY_THB': df_JPY_THB,
    'DXY': df_DXY,
    # พันธบัตรและดอกเบี้ย
    'US_10Y': df_US_10Y,
    'US_30Y': df_US_30Y,
    'TH_10Y': df_TH_10Y,
    'TH_20Y': df_TH_20Y,
    'TH_RATE': df_TH_RATE,
    # พลังงานและสินค้าโภคภัณฑ์
    'WTI': df_WTI,
    'Brent': df_เบรนท์,
    'BDI': df_BDI,
    # ดัชนีตลาดทุนและความผันผวน
    'SET50': df_SET50,
    'SP500': df_SP500,
    'SOX': df_SOX,
    'VIX': df_VIX
}

# 1. คำนวณ Correlation Matrix
corr_import = compute_stock_macro_correlation(import_stocks_dict, macro_factors_dict)

# คำนวณพอร์ตโฟลิโอเฉลี่ย (Import_Avg) และค่าเฉลี่ยรายหุ้น
corr_import_with_avg = corr_import.copy()
corr_import_with_avg.loc['Import_Avg'] = corr_import.mean(axis=0)
corr_import_with_avg['Mean_Corr'] = corr_import_with_avg.mean(axis=1)

print("=== ตาราง Correlation Matrix: กลุ่มหุ้นนำเข้า (Import Stocks) กับ 17 ปัจจัยมหภาค ===")
display(corr_import_with_avg.round(4))

# บันทึกตารางลงไฟล์ CSV
table_dir = Path('outputs/tables') if Path('outputs/tables').exists() else Path('../outputs/tables')
table_dir.mkdir(parents=True, exist_ok=True)
corr_import_with_avg.to_csv(table_dir / 'corr_matrix_import_macro.csv')

# 2. พล็อต Heatmap แสดงผลความสัมพันธ์
fig_dir = Path('outputs/figures') if Path('outputs/figures').exists() else Path('../outputs/figures')
fig_dir.mkdir(parents=True, exist_ok=True)

plt.figure(figsize=(16, 7.5))
sns.heatmap(
    corr_import,
    annot=True,
    cmap='coolwarm',
    vmin=-0.4,
    vmax=0.7,
    center=0,
    fmt=".3f",
    linewidths=0.8,
    cbar_kws={'label': 'Correlation Coefficient'}
)

plt.title("แผนภูมิความร้อน (Correlation Heatmap): หุ้นกลุ่มนำเข้า (10 ตัว) กับ 17 ปัจจัยมหภาคและดัชนีตลาดโลก", fontproperties=thai_font, fontsize=15, fontweight='bold', pad=15)
plt.xlabel("ปัจจัยมหภาคและดัชนีตลาด (Macroeconomic & Market Factors)", fontproperties=thai_font, fontsize=12)
plt.ylabel("หุ้นกลุ่มนำเข้า (Import Stocks)", fontproperties=thai_font, fontsize=12)
plt.xticks(fontproperties=thai_font, fontsize=10, rotation=45, ha='right')
plt.yticks(fontproperties=thai_font, fontsize=11, rotation=0)

cbar = plt.gca().collections[0].colorbar
cbar.ax.yaxis.label.set_fontproperties(thai_font)
cbar.ax.yaxis.label.set_size(11)

plt.tight_layout()
plt.savefig(fig_dir / 'heatmap_import_macro.png', dpi=300)
plt.show()


### 2. สหสัมพันธ์ของปัจจัยมหภาค กับ หุ้นกลุ่มส่งออก (Export Stocks)
การวิเคราะห์ความสัมพันธ์แบบจับคู่ (Pairwise Correlation) ระหว่างผลตอบแทนรายวัน (% Change) ของหุ้นกลุ่มส่งออก 10 ตัว กับ 17 ปัจจัยมหภาคและดัชนีตลาดโลก

In [ ]:
# รายชื่อกลุ่มหุ้นส่งออก 10 ตัว
export_stocks_dict = {
    'DELTA': df_DELTA, 'HANA': df_HANA, 'KCE': df_KCE, 'CPF': df_CPF, 'TU': df_TU,
    'AAI': df_AAI, 'STA': df_STA, 'BDMS': df_BDMS, 'BH': df_BH, 'ITC': df_ITC
}

# 1. คำนวณ Correlation Matrix
corr_export = compute_stock_macro_correlation(export_stocks_dict, macro_factors_dict)

# คำนวณพอร์ตโฟลิโอเฉลี่ย (Export_Avg) และค่าเฉลี่ยรายหุ้น
corr_export_with_avg = corr_export.copy()
corr_export_with_avg.loc['Export_Avg'] = corr_export.mean(axis=0)
corr_export_with_avg['Mean_Corr'] = corr_export_with_avg.mean(axis=1)

print("=== ตาราง Correlation Matrix: กลุ่มหุ้นส่งออก (Export Stocks) กับ 17 ปัจจัยมหภาค ===")
display(corr_export_with_avg.round(4))

# บันทึกตารางลงไฟล์ CSV
table_dir = Path('outputs/tables') if Path('outputs/tables').exists() else Path('../outputs/tables')
table_dir.mkdir(parents=True, exist_ok=True)
corr_export_with_avg.to_csv(table_dir / 'corr_matrix_export_macro.csv')

# 2. พล็อต Heatmap แสดงผลความสัมพันธ์
fig_dir = Path('outputs/figures') if Path('outputs/figures').exists() else Path('../outputs/figures')
fig_dir.mkdir(parents=True, exist_ok=True)

plt.figure(figsize=(16, 7.5))
sns.heatmap(
    corr_export,
    annot=True,
    cmap='coolwarm',
    vmin=-0.4,
    vmax=0.7,
    center=0,
    fmt=".3f",
    linewidths=0.8,
    cbar_kws={'label': 'Correlation Coefficient'}
)

plt.title("แผนภูมิความร้อน (Correlation Heatmap): หุ้นกลุ่มส่งออก (10 ตัว) กับ 17 ปัจจัยมหภาคและดัชนีตลาดโลก", fontproperties=thai_font, fontsize=15, fontweight='bold', pad=15)
plt.xlabel("ปัจจัยมหภาคและดัชนีตลาด (Macroeconomic & Market Factors)", fontproperties=thai_font, fontsize=12)
plt.ylabel("หุ้นกลุ่มส่งออก (Export Stocks)", fontproperties=thai_font, fontsize=12)
plt.xticks(fontproperties=thai_font, fontsize=10, rotation=45, ha='right')
plt.yticks(fontproperties=thai_font, fontsize=11, rotation=0)

cbar = plt.gca().collections[0].colorbar
cbar.ax.yaxis.label.set_fontproperties(thai_font)
cbar.ax.yaxis.label.set_size(11)

plt.tight_layout()
plt.savefig(fig_dir / 'heatmap_export_macro.png', dpi=300)
plt.show()


## Correlation Matrix (20 หุ้น + 2 พอร์ตเฉลี่ย Import/Export x 17 ปัจจัยมหภาคและดัชนีตลาดโลก)


In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from pathlib import Path
from IPython.display import display

# ฟังก์ชันดึงและทำความสะอาด Pct_Change เพื่อคำนวณ Correlation แบบจับคู่ (Pairwise)
def get_numeric_pct(df):
    df_clean = df.copy()
    df_clean['Date'] = pd.to_datetime(df_clean['Date'])
    if 'Pct_Change' in df_clean.columns:
        if df_clean['Pct_Change'].dtype == 'object':
            df_clean['Pct_Val'] = df_clean['Pct_Change'].str.rstrip('%').astype('float') / 100.0
        else:
            df_clean['Pct_Val'] = pd.to_numeric(df_clean['Pct_Change'], errors='coerce')
    else:
        df_clean = df_clean.sort_values('Date')
        df_clean['Pct_Val'] = df_clean['Close'].pct_change()
    return df_clean[['Date', 'Pct_Val']].dropna()

# รายชื่อกลุ่มหุ้น
import_stocks_dict = {
    'PTTGC': df_PTTGC, 'TOP': df_TOP, 'IRPC': df_IRPC, 'IVL': df_IVL, 'COM7': df_COM7,
    'BGRIM': df_BGRIM, 'GPSC': df_GPSC, 'EGCO': df_EGCO, 'RATCH': df_RATCH, 'BCP': df_BCP
}

export_stocks_dict = {
    'DELTA': df_DELTA, 'HANA': df_HANA, 'KCE': df_KCE, 'CPF': df_CPF, 'TU': df_TU,
    'AAI': df_AAI, 'STA': df_STA, 'BDMS': df_BDMS, 'BH': df_BH, 'ITC': df_ITC
}

# รายชื่อปัจจัยภายนอกและดัชนีตลาดโลก 17 ตัว
macro_factors_dict = {
    'USD_THB': df_USD_THB,
    'CNY_THB': df_CNY_THB,
    'EUR_THB': df_EUR_THB,
    'JPY_THB': df_JPY_THB,
    'DXY': df_DXY,
    'US_10Y': df_US_10Y,
    'US_30Y': df_US_30Y,
    'TH_10Y': df_TH_10Y,
    'TH_20Y': df_TH_20Y,
    'TH_RATE': df_TH_RATE,
    'WTI': df_WTI,
    'Brent': df_เบรนท์,
    'BDI': df_BDI,
    'SET50': df_SET50,
    'SP500': df_SP500,
    'SOX': df_SOX,
    'VIX': df_VIX
}

# เก็บข้อมูลผลลัพธ์
scatter_corr_results = []

# 1. คำนวณฝั่งหุ้นนำเข้า
for stock_name, df_stock in import_stocks_dict.items():
    df_s_clean = get_numeric_pct(df_stock)
    for factor_name, df_factor in macro_factors_dict.items():
        df_f_clean = get_numeric_pct(df_factor)
        merged = pd.merge(df_s_clean, df_f_clean, on='Date', how='inner')
        corr_val = merged['Pct_Val_x'].corr(merged['Pct_Val_y']) if len(merged) >= 5 else np.nan
        scatter_corr_results.append({
            'Stock': stock_name,
            'Type': 'หุ้นนำเข้า',
            'Macro_Factor': factor_name,
            'Scatter_Correlation': corr_val
        })

# 2. คำนวณฝั่งหุ้นส่งออก
for stock_name, df_stock in export_stocks_dict.items():
    df_s_clean = get_numeric_pct(df_stock)
    for factor_name, df_factor in macro_factors_dict.items():
        df_f_clean = get_numeric_pct(df_factor)
        merged = pd.merge(df_s_clean, df_f_clean, on='Date', how='inner')
        corr_val = merged['Pct_Val_x'].corr(merged['Pct_Val_y']) if len(merged) >= 5 else np.nan
        scatter_corr_results.append({
            'Stock': stock_name,
            'Type': 'หุ้นส่งออก',
            'Macro_Factor': factor_name,
            'Scatter_Correlation': corr_val
        })

# แปลงเป็น DataFrame
df_scatter_corr = pd.DataFrame(scatter_corr_results)

# Pivot ตารางให้อยู่ในรูปแบบ Matrix
matrix_scatter_corr = df_scatter_corr.pivot(index='Stock', columns='Macro_Factor', values='Scatter_Correlation')

# 3. คำนวณค่าเฉลี่ยของพอร์ตนำเข้า (Import_Avg) และพอร์ตส่งออก (Export_Avg)
import_names = list(import_stocks_dict.keys())
export_names = list(export_stocks_dict.keys())
ordered_factors = list(macro_factors_dict.keys())

import_avg_series = matrix_scatter_corr.loc[import_names].mean(axis=0)
export_avg_series = matrix_scatter_corr.loc[export_names].mean(axis=0)

matrix_scatter_corr.loc['Import_Avg'] = import_avg_series
matrix_scatter_corr.loc['Export_Avg'] = export_avg_series

# เพิ่ม Import_Avg และ Export_Avg ลงใน df_scatter_corr สำหรับขั้นตอน Deep-dive
for f_name in ordered_factors:
    scatter_corr_results.append({
        'Stock': 'Import_Avg',
        'Type': 'พอร์ตนำเข้าเฉลี่ย',
        'Macro_Factor': f_name,
        'Scatter_Correlation': import_avg_series[f_name]
    })
    scatter_corr_results.append({
        'Stock': 'Export_Avg',
        'Type': 'พอร์ตส่งออกเฉลี่ย',
        'Macro_Factor': f_name,
        'Scatter_Correlation': export_avg_series[f_name]
    })
df_scatter_corr = pd.DataFrame(scatter_corr_results)

# จัดเรียงลำดับแถวใหม่: นำเข้า 10 ตัว + Import_Avg และ ส่งออก 10 ตัว + Export_Avg
ordered_stocks_with_avg = import_names + ['Import_Avg'] + export_names + ['Export_Avg']
matrix_scatter_corr = matrix_scatter_corr.reindex(index=ordered_stocks_with_avg, columns=ordered_factors)

# เพิ่มคอลัมน์ Mean_Corr เพื่อดูค่าเฉลี่ยทุกปัจจัย
matrix_scatter_corr['Mean_Corr'] = matrix_scatter_corr.mean(axis=1)

print("=== ตารางค่า Correlation ครบทั้ง 20 หุ้น + 2 พอร์ตเฉลี่ย (Import_Avg & Export_Avg) x 17 ปัจจัยมหภาค ===")
display(matrix_scatter_corr.round(4))

# บันทึกตารางลงไฟล์ CSV
table_dir = Path('outputs/tables') if Path('outputs/tables').exists() else Path('../outputs/tables')
table_dir.mkdir(parents=True, exist_ok=True)
matrix_scatter_corr.to_csv(table_dir / 'corr_matrix_master_all.csv')

# 4. สร้างกราฟ Heatmap รวม (22 แถว x 17 ปัจจัย)
thai_font = fm.FontProperties(fname='./Sarabun-Regular.ttf')
sns.set_theme(style="white")

plt.figure(figsize=(18, 13))
# ไม่แสดงคอลัมน์ Mean_Corr ใน Heatmap เพื่อเน้นเฉพาะ 17 ปัจจัย
heatmap_data = matrix_scatter_corr[ordered_factors]

sns.heatmap(
    heatmap_data,
    annot=True,
    cmap='coolwarm',
    vmin=-0.5,
    vmax=0.7,
    center=0,
    fmt=".3f",
    linewidths=0.5,
    cbar_kws={'label': 'Correlation Coefficient'}
)

plt.title("Heatmap: สหสัมพันธ์ (Pairwise Correlation) ครบวงจร 20 หุ้น + 2 พอร์ตเฉลี่ย\n(รวมแถว Import_Avg และ Export_Avg เทียบกับ 17 ปัจจัยมหภาคและดัชนีตลาด)", fontproperties=thai_font, fontsize=16, fontweight='bold', pad=20)
plt.xticks(fontproperties=thai_font, fontsize=11, rotation=45, ha='right')
plt.yticks(fontproperties=thai_font, fontsize=11, rotation=0)

ax = plt.gca()
cbar = ax.collections[0].colorbar
cbar.ax.yaxis.label.set_fontproperties(thai_font)
cbar.ax.yaxis.label.set_size(12)

# บันทึกภาพ Heatmap ลงโฟลเดอร์ outputs/figures/
fig_dir = Path('outputs/figures') if Path('outputs/figures').exists() else Path('../outputs/figures')
fig_dir.mkdir(parents=True, exist_ok=True)
plt.tight_layout()
plt.savefig(fig_dir / 'heatmap_master_all.png', dpi=300)
plt.show()


### บทวิเคราะห์เปรียบเทียบความสัมพันธ์กับราคาน้ำมัน (WTI & Brent) ระหว่างหุ้นนำเข้าและส่งออก

วิเคราะห์ความแตกต่างเชิงปริมาณของค่าความสัมพันธ์แบบรายคู่ปัจจัย (Pairwise Correlation) ของกลุ่มหุ้นนำเข้า 10 ตัว และหุ้นส่งออก 10 ตัว เทียบกับราคาน้ำมันทั้ง WTI และ Brent

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.font_manager as fm

# ดึงข้อมูลจาก df_scatter_corr ที่คำนวณไว้ในขั้นตอนก่อนหน้า
# กรองเฉพาะปัจจัยราคาน้ำมัน WTI และ Brent
df_oil_corr = df_scatter_corr[df_scatter_corr['Macro_Factor'].isin(['WTI', 'Brent'])].copy()

# 1. คำนวณค่าสถิติเชิงพรรณนาจำแนกตามประเภทหุ้นและราคาน้ำมัน
oil_summary = df_oil_corr.groupby(['Type', 'Macro_Factor'])['Scatter_Correlation'].agg(['mean', 'std', 'min', 'max']).reset_index()

print("=== ตารางสรุปค่าเฉลี่ยและช่วงค่า Correlation จำแนกตามกลุ่ม ===")
display(oil_summary)

# 2. ทำการพล็อตกราฟเปรียบเทียบความแตกต่างจำแนกรายหุ้น
thai_font = fm.FontProperties(fname='./Sarabun-Regular.ttf')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6), sharey=True)

# กรองข้อมูลรายตัวเพื่อพล็อต
import_oil = df_oil_corr[df_oil_corr['Type'] == 'หุ้นนำเข้า']
export_oil = df_oil_corr[df_oil_corr['Type'] == 'หุ้นส่งออก']

# พล็อตฝั่งหุ้นนำเข้า
sns.barplot(data=import_oil, x='Stock', y='Scatter_Correlation', hue='Macro_Factor', ax=ax1, palette='Greens_d')
ax1.set_title('ค่าความสัมพันธ์ของ "หุ้นกลุ่มนำเข้า" กับราคาน้ำมัน', fontproperties=thai_font, fontsize=12, fontweight='bold')
ax1.set_xlabel('รายชื่อหุ้นนำเข้า', fontproperties=thai_font, fontsize=10)
ax1.set_ylabel('Correlation Coefficient', fontproperties=thai_font, fontsize=10)
ax1.set_xticklabels(ax1.get_xticklabels(), rotation=45)
ax1.axhline(0, color='grey', linestyle='--', linewidth=0.8)

# พล็อตฝั่งหุ้นส่งออก
sns.barplot(data=export_oil, x='Stock', y='Scatter_Correlation', hue='Macro_Factor', ax=ax2, palette='Purples_d')
ax2.set_title('ค่าความสัมพันธ์ของ "หุ้นกลุ่มส่งออก" กับราคาน้ำมัน', fontproperties=thai_font, fontsize=12, fontweight='bold')
ax2.set_xlabel('รายชื่อหุ้นส่งออก', fontproperties=thai_font, fontsize=10)
ax2.set_ylabel('', fontproperties=thai_font, fontsize=10)
ax2.set_xticklabels(ax2.get_xticklabels(), rotation=45)
ax2.axhline(0, color='grey', linestyle='--', linewidth=0.8)

fig.suptitle('การเปรียบเทียบค่าความสัมพันธ์เชิงเส้นจำแนกรายตัวระหว่างกลุ่มนำเข้าและส่งออกต่อราคาน้ำมัน', fontproperties=thai_font, fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


### บทวิเคราะห์เปรียบเทียบดัชนีเซมิคอนดักเตอร์โลก (SOX) กับหุ้นกลุ่มอิเล็กทรอนิกส์

วิเคราะห์ความเชื่อมโยงของวัฏจักรชิ้นส่วนอิเล็กทรอนิกส์ระดับโลกผ่านดัชนี Philadelphia Semiconductor (SOX) เทียบกับหุ้นอิเล็กทรอนิกส์ไทย (DELTA, HANA, KCE) และหุ้นทั้งพอร์ต

In [ ]:
# บทวิเคราะห์เจาะลึก: ดัชนีเซมิคอนดักเตอร์โลก (SOX) กับหุ้นกลุ่มอิเล็กทรอนิกส์และเทคโนโลยี
df_sox_corr = df_scatter_corr[df_scatter_corr['Macro_Factor'] == 'SOX'].copy()

plt.figure(figsize=(15, 5))
thai_font = fm.FontProperties(fname='./Sarabun-Regular.ttf')

# จัดเรียงตามค่า Correlation จากมากไปน้อย
df_sox_sorted = df_sox_corr.sort_values(by='Scatter_Correlation', ascending=False)
palette = ['#2ca02c' if t == 'หุ้นนำเข้า' else '#1f77b4' for t in df_sox_sorted['Type']]

ax = sns.barplot(data=df_sox_sorted, x='Stock', y='Scatter_Correlation', palette=palette)
ax.set_title('ค่าความสัมพันธ์ของหุ้น 20 ตัว กับดัชนีเซมิคอนดักเตอร์โลก (SOX)\n(สังเกต: HANA, KCE, COM7 มีค่าความสัมพันธ์เชิงบวกสูงสุดอย่างมีนัยสำคัญ)', fontproperties=thai_font, fontsize=13, fontweight='bold')
ax.set_xlabel('ชื่อหุ้น (สีเขียว: หุ้นนำเข้า / สีฟ้า: หุ้นส่งออก)', fontproperties=thai_font, fontsize=11)
ax.set_ylabel('Correlation กับ SOX', fontproperties=thai_font, fontsize=11)
ax.axhline(0, color='grey', linestyle='--', linewidth=0.8)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print("=== สรุปค่า Correlation กับ SOX เรียงจากมากไปน้อย ===")
display(df_sox_sorted[['Stock', 'Type', 'Scatter_Correlation']].reset_index(drop=True))


### บทวิเคราะห์เปรียบเทียบดัชนีตลาดหุ้นไทย (SET50) และดัชนีความผันผวนโลก (VIX)

วิเคราะห์ความไวต่อสภาวะตลาดในประเทศ (Market Beta) ผ่าน SET50 ควบคู่กับผลกระทบจากความตื่นตระหนกของตลาดการเงินโลกผ่านดัชนีความกลัว (VIX: Fear Index)

In [ ]:
# บทวิเคราะห์เจาะลึก: ดัชนีตลาดหุ้นไทย (SET50) และดัชนีความผันผวนของโลก (VIX)
df_set_vix = df_scatter_corr[df_scatter_corr['Macro_Factor'].isin(['SET50', 'VIX'])].copy()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5), sharey=False)
thai_font = fm.FontProperties(fname='./Sarabun-Regular.ttf')

# 1. SET50
df_set50 = df_scatter_corr[df_scatter_corr['Macro_Factor'] == 'SET50'].sort_values(by='Scatter_Correlation', ascending=False)
sns.barplot(data=df_set50, x='Stock', y='Scatter_Correlation', hue='Type', ax=ax1, palette={'หุ้นนำเข้า': '#2ca02c', 'หุ้นส่งออก': '#1f77b4'})
ax1.set_title('ความสัมพันธ์กับดัชนี SET50 (Market Beta)', fontproperties=thai_font, fontsize=12, fontweight='bold')
ax1.set_xlabel('ชื่อหุ้น', fontproperties=thai_font, fontsize=10)
ax1.set_ylabel('Correlation กับ SET50', fontproperties=thai_font, fontsize=10)
ax1.set_xticklabels(ax1.get_xticklabels(), rotation=45)
ax1.axhline(0, color='grey', linestyle='--', linewidth=0.8)

# 2. VIX
df_vix = df_scatter_corr[df_scatter_corr['Macro_Factor'] == 'VIX'].sort_values(by='Scatter_Correlation', ascending=True)
sns.barplot(data=df_vix, x='Stock', y='Scatter_Correlation', hue='Type', ax=ax2, palette={'หุ้นนำเข้า': '#2ca02c', 'หุ้นส่งออก': '#1f77b4'})
ax2.set_title('ความสัมพันธ์กับดัชนีความผันผวนโลก (VIX: Fear Index)', fontproperties=thai_font, fontsize=12, fontweight='bold')
ax2.set_xlabel('ชื่อหุ้น', fontproperties=thai_font, fontsize=10)
ax2.set_ylabel('Correlation กับ VIX', fontproperties=thai_font, fontsize=10)
ax2.set_xticklabels(ax2.get_xticklabels(), rotation=45)
ax2.axhline(0, color='grey', linestyle='--', linewidth=0.8)

fig.suptitle('การวิเคราะห์อิทธิพลของตลาดในประเทศ (SET50) และสภาวะความเสี่ยงโลก (VIX)', fontproperties=thai_font, fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


#### สรุปผลการวิเคราะห์และประเด็นสำคัญ (Key Insights ครบ 17 ปัจจัยมหภาคและดัชนีตลาด):

1. **ดัชนีตลาดหุ้นในประเทศ (SET50)**:
   * มีค่าสหสัมพันธ์เชิงบวกสูงสุดต่อพอร์ตหุ้นไทยทั้ง 2 กลุ่ม โดยเฉพาะ **หุ้นกลุ่มนำเข้า** มี Correlation สูงถึง **0.55 ถึง 0.68** (เช่น EGCO 0.68, IRPC 0.64, RATCH 0.64, GPSC 0.64, PTTGC 0.64) สะท้อนความเป็นหุ้น Large-Cap ที่อิงกับทิศทางตลาดไทย
   * หุ้นส่งออกส่วนใหญ่มีความสัมพันธ์ปานกลาง (0.20 ถึง 0.51) ขณะที่ **DELTA** มี Correlation กับ SET50 ต่ำเป็นพิเศษ (+0.076) เนื่องจากเคลื่อนไหวตามรอบปัจจัยเฉพาะตัวสูง

2. **ดัชนีกลุ่มอุตสาหกรรมเฉพาะ (SOX) และหุ้นอิเล็กทรอนิกส์**:
   * **SOX (Philadelphia Semiconductor Index)** มี Correlation เชิงบวกสูงมากกับหุ้นอิเล็กทรอนิกส์อย่าง **HANA (0.417)** และ **KCE (0.374)** รวมถึง COM7 (0.329)
   * แสดงให้เห็นชัดเจนว่าซัพพลายเชนชิปโลกเป็นตัวแปรที่มีพลังในการทำนายผลตอบแทนของกลุ่มอิเล็กทรอนิกส์ไทยมากกว่าปัจจัยมหภาคทั่วไป

3. **ดัชนีความผันผวนของโลก (VIX)**:
   * มี **สหสัมพันธ์เป็นลบ (Negative Correlation)** ต่อหุ้นเกือบทุกตัวในพอร์ตอย่างมีนัยสำคัญ (-0.15 ถึง -0.30)
   * เมื่อความเสี่ยงหรือความกลัวในตลาดการเงินโลกพุ่งสูงขึ้น เงินทุนมักจะไหลออกจากตลาดเกิดใหม่ (Emerging Markets) และกดดันราคาหุ้นไทยในภาพรวม

4. **ราคาน้ำมันโลก (WTI & Brent)**:
   * มีผลบวกชัดเจนต่อกลุ่มพลังงานและปิโตรเคมีในกลุ่มนำเข้า (**PTTGC 0.46, TOP 0.46, IVL 0.40, IRPC 0.38, BCP 0.13**) จากผลกำไรสินค้าคงคลัง (Inventory Gain) และค่าการกลั่น
   * ในขณะที่กลุ่มโรงไฟฟ้าและสาธารณูปโภค (BGRIM, RATCH) และกลุ่มส่งออกอาหาร มีค่าความสัมพันธ์ใกล้ศูนย์หรือไม่ได้รับประโยชน์โดยตรงในระดับรายวัน

5. **อัตราแลกเปลี่ยน (USD/THB, CNY/THB, EUR/THB, JPY/THB, DXY)**:
   * **DXY (Dollar Index)**: มีความสัมพันธ์เป็นบวกกับหุ้นกลุ่มโรงพยาบาลและสัตว์เลี้ยงส่งออก เช่น **ITC (+0.36), AAI (+0.23), BDMS (+0.20), BH (+0.18)** เพราะการแข็งค่าของดอลลาร์ส่งผลดีต่อรายได้ส่งออกและผู้ป่วยต่างชาติ
   * **CNY/THB (หยวนจีน)**: มีความสัมพันธ์เชิงบวกกับกลุ่มส่งออกสินค้าที่พึ่งพาตลาดจีน เช่น **KCE (+0.35), HANA (+0.30), TU (+0.24)** ในขณะที่มีความสัมพันธ์เป็นลบกับกลุ่มโรงไฟฟ้าสาธารณูปโภค

6. **พันธบัตรและอัตราดอกเบี้ย (Yields & Rates)**:
   * พันธบัตรสหรัฐฯ (`US_10Y`, `US_30Y`) มีความสัมพันธ์เชิงบวกระดับปานกลางกับกลุ่มพลังงาน (`TOP 0.41`, `BCP 0.40`)
   * ขณะที่อัตราดอกเบี้ยนโยบายไทย (`TH_RATE`) มีความสัมพันธ์ระดับรายวันค่อนข้างต่ำ ซึ่งสะท้อนว่านโยบายการเงินส่งผลกระทบแบบมีระยะเวลาหน่วง (Lag effect) ไม่ใช่การตอบสนองวันต่อวัน


In [ ]:
# ==============================================================================
# ตรวจสอบภาพรวมชุดข้อมูลทั้งหมดในโปรเจกต์ (Raw, Intermediate, Processed)
# สรุปจำนวนแถว (Rows), จำนวนคอลัมน์ (Cols), และรายชื่อหัวคอลัมน์ (Column Headers)
# ==============================================================================
import os
import csv
import zipfile
from pathlib import Path
from xml.etree import ElementTree as ET
import pandas as pd
from IPython.display import display

# ตรวจสอบ path ของโฟลเดอร์ data ทั้งเมื่อรันจาก root หรือจากโฟลเดอร์ notebooks/
data_candidates = [Path("../data"), Path("data"), Path("./data")]
data_path = next((p for p in data_candidates if p.exists()), None)

if data_path is None:
    print("❌ ไม่พบโฟลเดอร์ data/")
else:
    def inspect_csv(file_path):
        encodings = ["utf-8", "utf-8-sig", "tis-620", "cp874", "latin1"]
        for enc in encodings:
            try:
                with open(file_path, "r", encoding=enc) as f:
                    reader = csv.reader(f)
                    header = next(reader, None)
                    if header is None:
                        return 0, []
                    header = [h.strip().replace("﻿", "").strip('"') for h in header]
                    count = sum(1 for _ in reader)
                    return count, header
            except UnicodeDecodeError:
                continue
            except Exception as e:
                return 0, [f"ERR: {e}"]
        return 0, ["Encoding error"]

    def inspect_xlsx(file_path):
        try:
            with zipfile.ZipFile(file_path) as z:
                shared_strings = []
                if "xl/sharedStrings.xml" in z.namelist():
                    tree = ET.fromstring(z.read("xl/sharedStrings.xml"))
                    for si in tree.findall("{http://schemas.openxmlformats.org/spreadsheetml/2006/main}si"):
                        t = si.find("{http://schemas.openxmlformats.org/spreadsheetml/2006/main}t")
                        if t is not None:
                            shared_strings.append(t.text or "")
                        else:
                            r_ts = si.findall(".//{http://schemas.openxmlformats.org/spreadsheetml/2006/main}t")
                            shared_strings.append("".join(rt.text or "" for rt in r_ts))
                sheet_tree = ET.fromstring(z.read("xl/worksheets/sheet1.xml"))
                rows = sheet_tree.findall("{http://schemas.openxmlformats.org/spreadsheetml/2006/main}sheetData/{http://schemas.openxmlformats.org/spreadsheetml/2006/main}row")
                if not rows:
                    return 0, []
                header_cells = rows[0].findall("{http://schemas.openxmlformats.org/spreadsheetml/2006/main}c")
                header = []
                for c in header_cells:
                    t = c.attrib.get("t")
                    v = c.find("{http://schemas.openxmlformats.org/spreadsheetml/2006/main}v")
                    val = v.text if v is not None else ""
                    if t == "s" and val.isdigit():
                        header.append(shared_strings[int(val)])
                    else:
                        header.append(val)
                return len(rows) - 1, header
        except Exception as e:
            return 0, [f"ERR: {e}"]

    sections = [
        ("01_raw", "1. ข้อมูลดิบ (Raw Data)"),
        ("02_intermediate", "2. ข้อมูลคลีนแล้ว (Intermediate Cleaned Data)"),
        ("03_processed", "3. ข้อมูลประมวลผลขั้นสุดท้าย (Final Processed Data)")
    ]

    all_summary = []
    
    for folder, title in sections:
        sub_dir = data_path / folder
        if not sub_dir.exists():
            continue
        
        file_records = []
        for root, dirs, files in os.walk(sub_dir, followlinks=False):
            for f in sorted(files):
                if f.endswith((".csv", ".xlsx")) and not f.startswith("."):
                    full_path = Path(root) / f
                    rel_to_sub = full_path.relative_to(sub_dir)
                    
                    if f.endswith(".csv"):
                        row_count, headers = inspect_csv(full_path)
                    else:
                        row_count, headers = inspect_xlsx(full_path)
                    
                    file_records.append({
                        "โฟลเดอร์ย่อย": str(rel_to_sub.parent) if str(rel_to_sub.parent) != "." else "-",
                        "ชื่อไฟล์": f,
                        "จำนวนแถว (Rows)": row_count,
                        "จำนวนคอลัมน์ (Cols)": len(headers),
                        "หัวคอลัมน์ (Column Headers)": ", ".join(headers)
                    })
                    all_summary.append({
                        "folder": folder,
                        "file": f,
                        "rows": row_count,
                        "cols": len(headers),
                        "headers": headers
                    })
        
        df_report = pd.DataFrame(file_records)
        print(f"{'='*100}📌 {title} (รวม {len(df_report)} ไฟล์){'='*100}")
        pd.set_option('display.max_columns', None)
        pd.set_option('display.max_colwidth', None)
        pd.set_option('display.width', 1000)
        display(df_report)

    total_files = len(all_summary)
    total_rows = sum(x["rows"] for x in all_summary)
    print(f"{'='*100}")
    print(f"🎉 ตรวจสอบครบถ้วนทุกชุดข้อมูล: รวมทั้งหมด {total_files} ไฟล์ | จำนวนแถวรวม {total_rows:,} แถว")
    print(f"{'='*100}")
